# Inputs

## User inputs

In [ ]:
# Set your version and round numbers here
version_nb = 9
round_nb = 1
SAVE_OUTPUTS = False # Export the results as figures and csv files
CLEAN = True # reduce the size of the dataframe, if one is concerned by memory issue. Debugging is made difficult with that option

scc_usd_p_tco2e = 0 # social cost of carbon; 280 Moore et al, https://tools.fourthtack.com/sda/


## Libraries

In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import zipfile
import random
import importlib
import utility_functions as utility_functions
importlib.reload(utility_functions)
from utility_functions import clean_labels_for_plotting, reduce_size_df
from default_assumptions import DEPRECIATION_TIME
import re
from matplotlib.lines import Line2D
import numpy as np
from matplotlib.patches import Patch
import gc

In [ ]:
# INPUT SCENARIOS FILE
scenarios_input_file = f"../output/fem_results/version_{version_nb}/results_summary.xlsx"
round_id = f"round_{round_nb}"

# OUTPUT RESULTS SAVE LOCATION:
RUN_OUTPUT_DIR = f"../output/fem_results/version_{version_nb}/zip_files"
RUN_OUTPUT_DIR = Path(RUN_OUTPUT_DIR)

figures_dir = Path(f"../output/fem_results/version_{version_nb}/figures")
figures_dir.mkdir(parents=True, exist_ok=True)

output_csv_dir = Path(f"../output/fem_results/version_{version_nb}/output_csv")
output_csv_dir.mkdir(parents=True, exist_ok=True)

## Functions

In [ ]:
line_styles = ['-', '--',(0,(3,5,1,5,1,5)), ':',(0, (1, 10))]
line_styles = [
    '-',                      # Solid line
    (0, (6, 3)),             # Dashed
    (0, (1, 1)),             # Densely dotted
    (0, (2, 2)),             # Short dashed
    (0, (6, 2, 1, 2)),       # Dash-dot
    (0, (1, 3)),             # Loosely dotted (dots with space)
    (0, (6, 2, 1, 2, 1, 2)), # Dash-dot-dot
]

def split_scenario_base_and_ramp_year(scenario_name: str):
    """Return cleaned scenario name and ramp-up year parsed from *_biofuel_ramp_up_YYYY."""
    s = str(scenario_name)
    match = re.search(r'_biofuel_ramp_up_(\d{4})$', s, flags=re.IGNORECASE)
    ramp_year = int(match.group(1)) if match else None
    base_name = re.sub(r'_biofuel_ramp_up_\d{4}$', '', s, flags=re.IGNORECASE)
    return base_name, ramp_year

def format_scenario_base_label(scenario_name: str) -> str:
    base_name, _ = split_scenario_base_and_ramp_year(scenario_name)
    return base_name 


def build_ramp_up_years_title(scenario_names, prefix='Biofuel ramp-up year'):
    years = sorted({
        ramp_year
        for _, ramp_year in (split_scenario_base_and_ramp_year(name) for name in scenario_names)
        if ramp_year is not None
    })
    if len(years) == 0:
        return prefix
    return f"{prefix}: {', '.join(str(y) for y in years)}"

def sort_scenario_name(p):
    name_part = p.name.split('_')[1]
    try:
        return (0, int(name_part))  # numeric first
    except ValueError:
        return (1, name_part)       # alphanumeric next

scenario_folders = sorted(
    [p for p in RUN_OUTPUT_DIR.glob('scenario_*') if p.is_dir()],
    key=sort_scenario_name
)

def read_first_existing_csv(zf, candidates):
    files_in_zip = set(zf.namelist())
    for path in candidates:
        if path in files_in_zip:
            return pd.read_csv(zf.open(path))
    return None


def cumulative_emissions_mt_for(scenario_name):
    """Cumulative CO2e emissions (million tonnes) for one scenario from starting_year onward."""
    sub = df_fuels_filtered_annual[
        (df_fuels_filtered_annual.scenario_name == scenario_name)
        & (df_fuels_filtered_annual.estimation_year >= starting_year)
    ]
    assert len(sub)>0, f"Empty df_fuels_filtered_annual for scenario {scenario_name} and starting_year {starting_year}"
    return sub['tonne_co2e'].sum() * pow(10, -6)


def znz_share_5pct_year_interp(scenario_name, threshold=0.05):
    """Year the e-fuel ('ZNZ') share of fuel consumption reaches `threshold`, linearly interpolated
    between the 3-yearly estimation_year grid points."""
    sub = df_fuel_long[df_fuel_long['scenario_name'] == scenario_name].copy()
    assert len(sub)>0, f"Empty df_fuels_filtered_annual for scenario {scenario_name} and starting_year {starting_year}"

    sub['is_efuel'] = sub['fuel'].str.contains("e-", case=False, na=False)
    share_by_year = sub.groupby('estimation_year').apply(
        lambda g: g.loc[g['is_efuel'], 'fuel_consumption_mj_year_cohort_excl_wind'].sum()
        / g['fuel_consumption_mj_year_cohort_excl_wind'].sum()
        if g['fuel_consumption_mj_year_cohort_excl_wind'].sum() > 0 else np.nan,
        include_groups=False,
    ).sort_index()
    years = share_by_year.index.to_numpy(dtype=float)
    shares = share_by_year.to_numpy(dtype=float)
    if len(shares) == 0 or shares.max() < threshold:
        return np.nan
    idx_above = np.flatnonzero(shares >= threshold)[0]
    if idx_above == 0:
        return float(years[0])
    y0, y1 = years[idx_above - 1], years[idx_above]
    s0, s1 = shares[idx_above - 1], shares[idx_above]
    frac = (threshold - s0) / (s1 - s0) if s1 != s0 else 0.0
    return float(y0 + frac * (y1 - y0))


def annual_fund_components_usd(scenario_name):
    """Per-year revenue, reward and fund-after-reward (raw USD) for one scenario. Shared by the
    '## Fund' plotting cell, fund_after_reward_ever_negative and cumulative_fund_after_reward_for."""
    su_ru2_match_threshold = 0.05
    cols = ['estimation_year', 'su_price', 'ru2_price', 'ru2_cost_usd', 'ru1_cost_usd', 'su_revenue_usd', 'reward_usd']
    sub = df_fuels_filtered_annual.loc[
        (df_fuels_filtered_annual.scenario_name == scenario_name)
        & (df_fuels_filtered_annual.estimation_year > starting_year),
        cols,
    ].copy()
    assert len(sub)>0, f"Empty df_fuels_filtered_annual for scenario {scenario_name} and starting_year {starting_year}"

    sub['ru2_cost_usd_net'] = 0.0
    mask = (sub.ru2_price != 0) & (abs(sub.su_price - sub.ru2_price) / sub.ru2_price < su_ru2_match_threshold)
    sub.loc[mask, 'ru2_cost_usd_net'] = sub.loc[mask, 'ru2_cost_usd'] - sub.loc[mask, 'su_revenue_usd']
    sub['revenue'] = sub['ru2_cost_usd_net'].clip(lower=0) + sub['ru1_cost_usd']
    sub['fund_after_reward'] = sub['revenue'] - sub['reward_usd']
    return sub.groupby('estimation_year')[['revenue', 'fund_after_reward', 'reward_usd']].sum().reset_index()


def cumulative_fund_after_reward_for(scenario_name):
    """Cumulative fund revenue after reward disbursement (billion USD) for one scenario."""
    annual = annual_fund_components_usd(scenario_name)
    if annual.empty:
        return np.nan
    return annual['fund_after_reward'].sum() * pow(10, -9)


def net_cost_excl_scc_usd_for(scenario_name):
    """Net cumulative cost vs BAU (USD) for one scenario, excluding the social cost of carbon term."""
    cols_excl_scc = [c for c in cum_cost_usd.columns if c != 'Social cost of carbon']
    return cum_cost_usd.loc[scenario_name, cols_excl_scc].sum() - cum_cost_usd.loc[bau_label, cols_excl_scc].sum()


def abatement_cost_usd_tco2e_for(scenario_name):
    """Carbon abatement cost vs BAU (USD per tCO2e saved) for one scenario."""
    emissions_saved_mt = cumulative_emissions_mt_for(bau_label) - cumulative_emissions_mt_for(scenario_name)
    assert pd.notna(emissions_saved_mt), f"Empty emission saved df for {scenario_name}"
    return net_cost_excl_scc_usd_for(scenario_name) / (emissions_saved_mt * pow(10, 6))


def peak_biofuel_ej_for(scenario_name):
    """Peak single-year biofuel consumption (EJ) between starting_year and 2050 for one scenario."""
    biofuel_pattern = r'^bio[-/]'
    sub = df_fuel_long[
        (df_fuel_long.scenario_name == scenario_name)
        & (df_fuel_long.estimation_year >= starting_year) & (df_fuel_long.estimation_year <= 2050)
        & (df_fuel_long['fuel'].str.contains(biofuel_pattern, case=False, na=False, regex=True))
    ]
    consumption_by_year = sub.groupby('estimation_year')['fuel_consumption_mj_year_cohort_excl_wind'].sum()
    if len(consumption_by_year) == 0:
        return 0
    return consumption_by_year.max() * pow(10, -12)


def base_fuel_family(engine_label):
    """Collapse an engine/prev_engine label to its underlying fuel family, ignoring wind-assist add-ons
    (e.g. 'DF LNG + wind' and 'df_ice_lng_+_kite' both -> 'lng'), so wind-only retrofits can be told
    apart from retrofits that actually change the ship's fuel-consuming engine."""
    s = str(engine_label).lower()
    for family in ('ammonia', 'methanol', 'lng'):
        if family in s:
            return family
    return 'conventional'


def peak_engine_retrofit_dwt_pct_for(scenario_name):
    """Peak single-year share of fleet DWT (%) retrofitted to a different engine fuel family, between
    starting_year and 2050, for one scenario. Wind-assist-only retrofits (same fuel family as
    prev_engine, e.g. 'conventional' -> 'conventional + wind') do not count."""
    sub = df_fleet[
        (df_fleet.scenario_name == scenario_name)
        & (df_fleet.game_year >= starting_year) & (df_fleet.game_year <= 2050)
    ]
    assert len(sub) > 0, f"Empty df_fleet for scenario {scenario_name}"
    total_dwt_by_year = sub.groupby('game_year')['dwt_cohort'].sum()

    retrofits = sub[sub['action'] == 'retrofit'].copy()
    if len(retrofits) == 0:
        return 0.0
    is_engine_change = (
        retrofits['engine'].apply(base_fuel_family) != retrofits['prev_engine'].apply(base_fuel_family)
    )
    retrofit_dwt_by_year = retrofits[is_engine_change].groupby('game_year')['dwt_cohort'].sum() / year_step
    pct_by_year = (retrofit_dwt_by_year / total_dwt_by_year).reindex(total_dwt_by_year.index).fillna(0) * 100
    return pct_by_year.max()


def interpolate_to_annual(df, year_col, group_cols, value_cols, flat_first_period=True):
    """
    Interpolate game-year data to annual values.
    - First period: flat (held constant at first game year value).
    - Subsequent periods: linear interpolation between consecutive game year values.
    - Last game year: included as a single endpoint.

    Args:
        df                : DataFrame with one row per (group x game year), already aggregated.
        year_col          : name of the integer year column.
        group_cols        : list of columns that define each group (e.g. ['scenario_name']).
        value_cols        : list of numeric columns to interpolate.
        flat_first_period : if True, the first period is held flat (no interpolation).
    Returns:
        DataFrame with one row per (group x annual year).
    """
    all_rows = []

    for keys, grp in df.groupby(group_cols):
        grp_sorted = grp.sort_values(year_col).reset_index(drop=True)
        game_years = grp_sorted[year_col].values
        n = len(game_years)

        # pandas 2.x returns tuple keys even for single-column groupby; normalise to tuple
        if not isinstance(keys, tuple):
            keys = (keys,)
        key_dict = dict(zip(group_cols, keys))

        for i in range(n):
            y_curr = int(game_years[i])
            vals_curr = grp_sorted.loc[i, value_cols].to_dict()

            if i == n - 1:
                # Last game year: single row
                all_rows.append({**key_dict, year_col: y_curr, **vals_curr})
            else:
                y_next = int(game_years[i + 1])
                vals_next = grp_sorted.loc[i + 1, value_cols].to_dict()
                for yr in range(y_curr, y_next):
                    if i == 0 and flat_first_period:
                        interp_vals = {col: vals_curr[col] for col in value_cols}
                    else:
                        frac = (yr - y_curr) / (y_next - y_curr)
                        interp_vals = {
                            col: vals_curr[col] + (vals_next[col] - vals_curr[col]) * frac
                            for col in value_cols
                        }
                    all_rows.append({**key_dict, year_col: yr, **interp_vals})

    return pd.DataFrame(all_rows)


## Load results files

In [ ]:
# Load all scenario results from zip files only
scenario_dfs = []

# ---- Fleet results (zip only) ----
for scenario_folder in scenario_folders:
    # print(scenario_folder)
    scenario_num = scenario_folder.name.split('_')[1]
    round_zip = scenario_folder / round_id / "fleet_model_results.zip"

    if not round_zip.exists():
        continue

    with zipfile.ZipFile(round_zip, 'r') as zf:
        print(round_zip)
        df_scenario = read_first_existing_csv(
            zf,
            [
                "df_results_nash.csv",
                "df_results_opt.csv",
                f"{round_id}/df_results_nash.csv",
                f"{round_id}/df_results_opt.csv",
            ],
        )

    if df_scenario is None:
        continue

    df_scenario['scenario_number'] = scenario_num
    scenario_dfs.append(df_scenario)

df_fleet = pd.concat(scenario_dfs, ignore_index=True)

# Format labels used in plots
df_fleet = clean_labels_for_plotting(df_fleet, ['engine'])
df_fleet.engine = df_fleet.engine.str.replace('rigid sail', 'wind').str.replace('kite', 'wind')

print(f"Loaded {len(scenario_dfs)} scenarios into df_fleet with {len(df_fleet)} total rows")
print(f"Scenarios: {df_fleet['scenario_name'].unique().tolist()}")

# build the cohort versions
df_fleet['dwt_cohort'] = df_fleet['dwt'] * df_fleet['real_ships_nb']

# ---- Fuel results (zip only) ----
fuel_dfs = []

for scenario_folder in scenario_folders:
    scenario_num = scenario_folder.name.split('_')[1]
    round_zip = scenario_folder / round_id / "fleet_model_results.zip"

    if not round_zip.exists():
        continue

    with zipfile.ZipFile(round_zip, 'r') as zf:
        df_fuel_scenario = read_first_existing_csv(
            zf,
            [
                "df_rcc_fuel_nash.csv",
                "df_rcc_fuel_opt.csv",
                f"{round_id}/df_rcc_fuel_nash.csv",
                f"{round_id}/df_rcc_fuel_opt.csv",
            ],
        )

    if df_fuel_scenario is None:
        continue

    df_fuel_scenario = clean_labels_for_plotting(df_fuel_scenario, ['engine', 'fuel'])
    df_fuel_scenario['estimation_year'] = (
        df_fuel_scenario['game_year'] + df_fuel_scenario['planning_year'] - 1
    )
    df_fuel_scenario['scenario_number'] = scenario_num
    if CLEAN:
        cols_to_keep = [
                'eu_compliance_cost_usd',
                'total_fuel_cost',
                'total_compliance_cost_usd',
                'reg_reg_class',
                'scenario_number',
                'ship_year_id',
                'planning_year',
                'overall_emissions_factor_wtw_gco2eq_mj', 
                'fuel_consumption_mj_year_cohort', 
                'main_fuel_consumption_mj_year_cohort',
                'pilot_fuel_consumption_mj_year_cohort',
                # 'fuel_intensity_below_direct_tco2eq_tlsfoeq',
                'fuel_savings',
                'estimation_year', 
                'game_year', 
                'dcf_id', 
                'fuel',
                'pilot_fuel', 
                'fuel_1',
                'fuel_2',
                'fuel_3',
                'share_fuel_1_final',
                'share_fuel_2_final',
                'share_fuel_3_final',
                'reward_usd',
                'tco_id',
                'ru2_cost_usd_tvslfoeq', 
                'ru1_cost_usd_tvslfoeq', 
                'su_revenue_usd_tvslfoeq', 
                'total_consumption_tlsfo_eq',
                'real_ships_nb', 
                'imo_type',
                'su_price', 
                'ru2_price',
                'reward_usd_tlsfoeq',
            ]
    else:
        cols_to_keep = df_fuel_scenario.columns
    fuel_dfs.append(df_fuel_scenario[cols_to_keep])

df_rrc_fuel = pd.concat(fuel_dfs, ignore_index=True)
if CLEAN:
    del fuel_dfs, df_fuel_scenario
    gc.collect()
    df_rrc_fuel = reduce_size_df(df_rrc_fuel)

check_len = len(df_rrc_fuel)
df_rrc_fuel = df_rrc_fuel.merge(
    df_fleet[['scenario_number', 'scenario_name']].drop_duplicates(),
    on='scenario_number',
    how='left'
)
assert check_len == len(df_rrc_fuel)
if CLEAN:
    df_rrc_fuel = reduce_size_df(df_rrc_fuel)

In [ ]:
# Build long-format fuel consumption table:
# columns -> tco_id, scenario_name, estimation_year, fuel, fuel_consumption_mj_year_cohort, fuel_savings

base_cols = ['tco_id', 'scenario_name', 'estimation_year', 'fuel', 'fuel_consumption_mj_year_cohort', 'fuel_savings']

# Build valid ship-option keys from fleet table (not dcf_id alone)
key_cols = ['ship_year_id', 'dcf_id']
if ('scenario_number' in df_fleet.columns) and ('scenario_number' in df_rrc_fuel.columns):
    key_cols = ['scenario_number'] + key_cols

missing_fleet = [c for c in key_cols if c not in df_fleet.columns]
missing_fuel = [c for c in key_cols if c not in df_rrc_fuel.columns]
assert len(missing_fleet) == 0, f"Missing key cols in df_fleet: {missing_fleet}"
assert len(missing_fuel) == 0, f"Missing key cols in df_rrc_fuel: {missing_fuel}"

df_valid_keys = df_fleet[key_cols].drop_duplicates()
df_fuels_filtered = df_rrc_fuel.merge(df_valid_keys, on=key_cols, how='inner')

df_fuels_filtered = df_fuels_filtered[
    df_fuels_filtered.game_year == df_fuels_filtered.estimation_year
].copy()

assert len(df_fuels_filtered) > 0, (
    f"No fuel rows after key filter. key_cols={key_cols}, "
    f"fleet_keys={len(df_valid_keys)}, fuel_rows={len(df_rrc_fuel)}"
)
if CLEAN:
    df_fuels_filtered = reduce_size_df(df_fuels_filtered)
    del df_valid_keys
    gc.collect()

# 1) Rows with explicit fuel split (fuel_1/2/3 + share_fuel_1/2/3_final)
df_split = df_fuels_filtered[df_fuels_filtered['fuel_1'].notna()].copy()

parts = []
for fuel_nb in [1, 2, 3]:
    part = df_split[['tco_id', 'scenario_name', 'estimation_year', f'fuel_{fuel_nb}', 'fuel_consumption_mj_year_cohort', 'fuel_savings']].copy()
    part = part.rename(columns={f'fuel_{fuel_nb}': 'fuel'})
    part['fuel_consumption_mj_year_cohort'] = (
        df_split[f'share_fuel_{fuel_nb}_final'].fillna(0) * df_split['fuel_consumption_mj_year_cohort']
    )
    parts.append(part)

df_split_long = pd.concat(parts, ignore_index=True)
df_split_long = df_split_long[
    df_split_long['fuel'].notna() & (df_split_long['fuel_consumption_mj_year_cohort'] > 0)
]
assert (
    abs(
        (
            df_split_long.fuel_consumption_mj_year_cohort.sum() 
            - df_split.fuel_consumption_mj_year_cohort.sum()
            )
        /max(
            df_split.fuel_consumption_mj_year_cohort.sum(),
            pow(10,-6))
        ) < 0.01
)
if CLEAN:
    df_split_long = reduce_size_df(df_split_long)
    del df_split
    gc.collect()

# 2) Rows without split: split into main and pilot where applicable
df_nonsplit = df_fuels_filtered[df_fuels_filtered['fuel_1'].isna()].copy()

# Main fuel part
df_nonsplit_main = df_nonsplit[['tco_id', 'scenario_name', 'estimation_year', 'fuel', 'main_fuel_consumption_mj_year_cohort', 'fuel_savings']].copy()
df_nonsplit_main = df_nonsplit_main.rename(columns = {'main_fuel_consumption_mj_year_cohort': 'fuel_consumption_mj_year_cohort'})

# Pilot fuel part (if available)
df_nonsplit_pilot = df_nonsplit[['tco_id', 'scenario_name', 'estimation_year', 'pilot_fuel', 'pilot_fuel_consumption_mj_year_cohort', 'fuel_savings']].copy()
df_nonsplit_pilot = df_nonsplit_pilot.rename(columns={
    'pilot_fuel': 'fuel',
    'pilot_fuel_consumption_mj_year_cohort': 'fuel_consumption_mj_year_cohort',
})
df_nonsplit_pilot = df_nonsplit_pilot[
    df_nonsplit_pilot['fuel'].notna() & (df_nonsplit_pilot['fuel_consumption_mj_year_cohort'] > 0)
]
df_nonsplit_long = pd.concat([df_nonsplit_main, df_nonsplit_pilot], ignore_index=True)

df_nonsplit_long = df_nonsplit_long[
    df_nonsplit_long['fuel'].notna() & (df_nonsplit_long['fuel_consumption_mj_year_cohort'] > 0)
]

# 3) Final long dataframe
df_fuel_long = pd.concat([df_nonsplit_long, df_split_long], ignore_index=True)
if CLEAN:
    df_split_long = reduce_size_df(df_split_long)
    del df_nonsplit_long
    gc.collect()

# Keep exactly requested columns
df_fuel_long = df_fuel_long[
    ['tco_id', 'scenario_name', 'estimation_year', 'fuel', 'fuel_consumption_mj_year_cohort', 'fuel_savings']
].reset_index(drop=True)

assert (
    abs(
        (
            df_fuel_long.fuel_consumption_mj_year_cohort.sum()
            - df_fuels_filtered.fuel_consumption_mj_year_cohort.sum()
        )
        / max(df_fuels_filtered.fuel_consumption_mj_year_cohort.sum(), pow(10, -6))
    ) < 0.01
)
df_fuel_long = clean_labels_for_plotting(df_fuel_long, ['fuel'])

# Create wind-adjusted consumption column for visualization
df_fuel_long['fuel_consumption_mj_year_cohort_excl_wind'] = (
    df_fuel_long['fuel_consumption_mj_year_cohort'] * (1 - df_fuel_long['fuel_savings'].fillna(0))
)
df_fuel_long = clean_labels_for_plotting(df_fuel_long, ['fuel'])

df_fuels_filtered['tonne_co2e'] = (
    df_fuels_filtered['overall_emissions_factor_wtw_gco2eq_mj'] 
    * df_fuels_filtered['fuel_consumption_mj_year_cohort']
    * pow(10, -6)
)

df_fuels_filtered['fuel_consumption_mj_year_cohort_excl_wind'] = df_fuels_filtered['fuel_consumption_mj_year_cohort'] * (1 - df_fuels_filtered['fuel_savings'].fillna(0))

In [ ]:
# Pre-compute fund-related dollar columns on df_fuels_filtered, then build df_fuels_filtered_annual.
# Done here (before the Emissions section) so both Emissions and Fund sections share the same
# annually-interpolated dataframe.

_fund_raw_cols = ['ru2_cost_usd', 'ru1_cost_usd', 'su_revenue_usd']
for col in _fund_raw_cols:
    df_fuels_filtered[col] = (
        df_fuels_filtered[col + '_tvslfoeq']
        * df_fuels_filtered['total_consumption_tlsfo_eq']
        * df_fuels_filtered['real_ships_nb']
    )
df_fuels_filtered['reward_usd'] = (
    df_fuels_filtered['reward_usd_tlsfoeq']
    * df_fuels_filtered['total_consumption_tlsfo_eq']
    * df_fuels_filtered['real_ships_nb']
)

# Aggregate to one row per (scenario_name, estimation_year) before interpolating
_interp_sum_cols = ['tonne_co2e', 'ru2_cost_usd', 'ru1_cost_usd', 'su_revenue_usd', 'reward_usd']
_interp_mean_cols = ['su_price', 'ru2_price']

df_fuels_agg = (
    df_fuels_filtered
    .groupby(['scenario_name', 'estimation_year'])
    .agg(
        **{col: (col, 'sum') for col in _interp_sum_cols},
        **{col: (col, 'mean') for col in _interp_mean_cols},
    )
    .reset_index()
)

# Interpolate game years → annual values
df_fuels_filtered_annual = interpolate_to_annual(
    df_fuels_agg,
    year_col='estimation_year',
    group_cols=['scenario_name'],
    value_cols=_interp_sum_cols + _interp_mean_cols,
)
print(
    f"df_fuels_filtered_annual: {len(df_fuels_filtered_annual)} rows, "
    f"years {df_fuels_filtered_annual.estimation_year.min()}–{df_fuels_filtered_annual.estimation_year.max()}"
)
if CLEAN:

    df_fuels_filtered_annual = reduce_size_df(df_fuels_filtered_annual)

    del df_fuels_agg    
    gc.collect()

## Scenarios choice

In [ ]:
# Get available scenarios:
scenarios = sorted(df_fleet['scenario_name'].unique())
scenarios

In [ ]:
# # Filter comparison plots to a single ramp-up year
year_step = 3
starting_year = 2025
baseline_year = starting_year


ramp_up_year = str(starting_year + 9)
scenarios_sub = [x for x in scenarios if ramp_up_year in x]
bau_label = f"No IMO policy_biofuel_ramp_up_{ramp_up_year}"
# bau_label = "No IMO policy_biofuel_ramp_up_3034"
# bau_label = f"Baseline_biofuel_ramp_up_{ramp_up_year}"
BASELINE_SCENARIO_NUMBER = df_fleet[df_fleet.scenario_name == bau_label].scenario_number.iloc[0]
# scenarios_sub = ['Full foresight','Myopic','All behavioural biases included']
# scenarios_sub = scenarios

assert len(scenarios_sub) >0
scenarios_sub

# Scenario comparison plots

## SU market

In [ ]:
# SU price by year — color = ramp-up year, linestyle = base scenario
import textwrap

fig, ax = plt.subplots(figsize=(8, 6))

su_data = (
    df_fleet[
        (df_fleet.su_price.notna())
        &(df_fleet.scenario_name.isin(scenarios))
        &(df_fleet.action!="scrap_not_replaced")
        &~(df_fleet.scenario_name.str.contains("levy"))
        &~(df_fleet.scenario_name.str.contains("Levy"))
        &~(df_fleet.scenario_name.str.contains("No IMO"))
    ][['game_year', 'scenario_name', 'su_price']]
    .drop_duplicates()
)

# Parse scenario_name → base scenario + ramp-up year
su_data['ramp_year'] = su_data['scenario_name'].str.extract(r'ramp_up_(\d{4})').astype(int)
su_data['base_scenario'] = su_data['scenario_name'].str.replace(r'_biofuel_ramp_up_\d{4}$', '', regex=True)
su_data['base_scenario'] = su_data['base_scenario'].apply(format_scenario_base_label)

# Calculate mean SU price for each (game_year, base_scenario, ramp_year) combination
su_data = su_data.groupby(['game_year', 'base_scenario', 'ramp_year'])['su_price'].mean().reset_index()

# Assign colours by ramp-up year and linestyles by base scenario
ramp_years_sorted = sorted(su_data['ramp_year'].unique())
base_scenarios = sorted(su_data['base_scenario'].unique())

cmap = plt.cm.get_cmap('tab10', len(ramp_years_sorted))
year_colors = {yr: cmap(i) for i, yr in enumerate(ramp_years_sorted)}
base_styles = {bs: line_styles[i % len(line_styles)] for i, bs in enumerate(base_scenarios)}

for (bs, yr), grp in su_data.groupby(['base_scenario', 'ramp_year']):
    grp_sorted = grp.sort_values('game_year')
    ax.plot(
        grp_sorted['game_year'], grp_sorted['su_price'],
        color=year_colors[yr], linestyle=base_styles[bs], linewidth=2,
    )

# Build a two-part legend: one for years (color), one for base scenarios (linestyle)
year_handles = [Line2D([], [], color=year_colors[yr], lw=2, label=str(yr)) for yr in ramp_years_sorted]
style_handles = [Line2D([], [], color='grey', lw=2, linestyle=base_styles[bs], label=bs)
                 for bs in base_scenarios]

leg1 = ax.legend(
    handles=style_handles, title='Policy scenario',
    loc='upper center', bbox_to_anchor=(0.5, -0.1),
    ncol=min(2, len(base_scenarios)), frameon=True, fontsize=9,
 )
ax.add_artist(leg1)
ax.legend(
    handles=year_handles, title='Biofuel price ramp-up year',
    loc='upper center', bbox_to_anchor=(0.5, -0.3),
    ncol=min(5, len(ramp_years_sorted)), frameon=True, fontsize=8,
)

ax.set_ylabel('SU price ($/tonne CO2e)')
ax.set_xlabel("")
ax.grid(alpha=0.3)
# fig.subplots_adjust(bottom=0.35)

plt.show()
if SAVE_OUTPUTS:
    fig.savefig(figures_dir / f"su_price_v{version_nb}_round_{round_nb}.png", dpi=300, bbox_inches='tight')

## By ship type

In [ ]:
# Fleet composition breakdown by scenario (share of total dwt)
n_scenarios = len(scenarios_sub)

# Create subplots - arrange in rows (e.g., 2 rows for multiple scenarios)
n_cols = min(3, n_scenarios)
n_rows = (n_scenarios + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows + 1.5))
fig.suptitle(build_ramp_up_years_title(scenarios_sub), fontweight='bold')

# Flatten axes array for easier iteration
if n_scenarios == 1:
    axes = [axes]
else:
    axes = axes.flatten() if isinstance(axes, np.ndarray) else [axes]

for idx, scenario in enumerate(scenarios_sub):
    ax = axes[idx]
    
    # Aggregate by year and engine
    df_scenario_imo_type = df_fleet[
        (df_fleet['scenario_name'] == scenario)
    ].groupby(['game_year', 'imo_type'])['dwt_cohort'].sum().unstack(fill_value=0)

    # Plot stacked area chart with grouped colors (no legend)
    df_scenario_imo_type.plot(
        kind='bar',
        stacked=True,
        ax=ax,
        lw=0,
        legend = False,
    )
    
    ax.set_title(format_scenario_base_label(scenario))
    ax.set_ylabel('Deadweight')
    ax.set_xlabel('')
    ax.grid(axis='y', alpha=0.3)
ax.legend()

# Hide unused subplots
for idx in range(n_scenarios, len(axes)):
    axes[idx].set_visible(False)


plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

In [ ]:
# Fleet composition breakdown by scenario (share of total dwt)
n_scenarios = len(scenarios_sub)

# Create subplots - arrange in rows (e.g., 2 rows for multiple scenarios)
n_cols = min(3, n_scenarios)
n_rows = (n_scenarios + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows + 1.5))

# Flatten axes array for easier iteration
if n_scenarios == 1:
    axes = [axes]
else:
    axes = axes.flatten() if isinstance(axes, np.ndarray) else [axes]

for idx, scenario in enumerate(scenarios_sub):
    ax = axes[idx]
    
    # Aggregate by year and engine
    df_scenario_imo_type = df_fleet[
        (df_fleet['scenario_name'] == scenario)
    ].groupby(['game_year', 'imo_type'])['real_ships_nb'].sum().unstack(fill_value=0)

    # Plot stacked area chart with grouped colors (no legend)
    df_scenario_imo_type.plot(
        kind='bar',
        stacked=True,
        ax=ax,
        lw=0,
        legend = False,
    )
    
    ax.set_title(format_scenario_base_label(scenario), fontweight='bold')
    ax.set_ylabel('# ships')
    ax.set_xlabel('')
    ax.grid(axis='y', alpha=0.3)
ax.legend()

# Hide unused subplots
for idx in range(n_scenarios, len(axes)):
    axes[idx].set_visible(False)


plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

## By action

In [ ]:
# Actions breakdown by scenario (share of total dwt)
n_scenarios = len(scenarios_sub)

# Create subplots - arrange in rows (e.g., 2 rows for multiple scenarios)
n_cols = min(3, n_scenarios)
n_rows = (n_scenarios + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows + 1.5))

# Flatten axes array for easier iteration
if n_scenarios == 1:
    axes = [axes]
else:
    axes = axes.flatten() if isinstance(axes, np.ndarray) else [axes]

df_fleet_clean = df_fleet.copy()
df_fleet_clean = clean_labels_for_plotting(df_fleet_clean, ["action"])

actions = [
    'keep', 'retrofit', 
    'newbuild', 'early scrap replaced by newbuild', 'newbuild to grow fleet',
    'early scrap to reduce fleet',  'scrap not replaced'
]

for idx, scenario in enumerate(scenarios_sub):
    ax = axes[idx]
    
    # Aggregate by year and engine
    sub = df_fleet_clean[
        (df_fleet_clean['scenario_name'] == scenario)
    ].groupby(['game_year', 'action'])['dwt_cohort'].sum().unstack(fill_value=0)
    missing_actions = [c for c in actions if c not in sub.columns]
    sub[missing_actions] = 0

    # Convert to share of total dwt (%)
    df_scenario_share = sub.div(sub.sum(axis=1), axis=0).fillna(0) * 100
    
    # Plot stacked area chart with grouped colors (no legend)
    df_scenario_share[actions].plot(
        kind='bar',
        stacked=True,
        ax=ax,
        legend=False,
        lw=0,
    )
    
    ax.set_title(format_scenario_base_label(scenario), fontweight='bold')
    ax.set_ylabel('Share of fleet deadweight (%)')
    ax.set_xlabel('')
    ax.grid(axis='y', alpha=0.3)

ax.legend()
# Hide unused subplots
for idx in range(n_scenarios, len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

if SAVE_OUTPUTS:
    # Save figure to figures folder
    fig.savefig(figures_dir / f"actions_v{version_nb}_round_{round_nb}.png", dpi=300, bbox_inches='tight')

In [ ]:
# Age at scrapping by year — color = ramp-up year, linestyle = base scenario
fig, ax = plt.subplots(figsize=(8, 6))

age_data = (
    df_fleet[
        ((df_fleet.action.isin(['demand_scrap_early','newbuild_early_scrap']))
        |(df_fleet.action.str.contains('newbuild')))
        &(df_fleet.scenario_name.isin(scenarios))
    ]
    .groupby(["game_year",'scenario_name'])
    .prev_vessel_age.mean()
    .reset_index()
)

if len(age_data) > 0:
    # Parse scenario_name → base scenario + ramp-up year
    age_data['ramp_year'] = age_data['scenario_name'].str.extract(r'ramp_up_(\d{4})').astype(int)
    age_data['base_scenario'] = age_data['scenario_name'].str.replace(r'_biofuel_ramp_up_\d{4}$', '', regex=True)
    
    # Assign colours by ramp-up year and line_styles by base scenario
    ramp_years_sorted = sorted(age_data['ramp_year'].unique())
    base_scenarios = sorted(age_data['base_scenario'].unique())
    
    cmap = plt.cm.get_cmap('tab10', len(ramp_years_sorted))
    year_colors = {yr: cmap(i) for i, yr in enumerate(ramp_years_sorted)}
    line_styles = ['-', '--', '-.', ':',(0, (1, 10))]
    base_styles = {bs: line_styles[i % len(line_styles)] for i, bs in enumerate(base_scenarios)}
    
    for (bs, yr), grp in age_data.groupby(['base_scenario', 'ramp_year']):
        grp_sorted = grp.sort_values('game_year')
        ax.plot(
            grp_sorted['game_year'], grp_sorted['prev_vessel_age'],
            color=year_colors[yr], linestyle=base_styles[bs], linewidth=2, marker='o',
        )
    
    # Build a two-part legend: one for years (color), one for base scenarios (linestyle)
    year_handles = [Line2D([], [], color=year_colors[yr], lw=2, label=str(yr)) for yr in ramp_years_sorted]
    style_handles = [Line2D([], [], color='grey', lw=2, linestyle=base_styles[bs], label=bs)
                     for bs in base_scenarios]
    
    leg1 = ax.legend(
        handles=style_handles, title='Policy scenario',
        loc='upper center', bbox_to_anchor=(0.5, -0.1),
        ncol=min(4, len(base_scenarios)), frameon=True, fontsize=9,
    )
    ax.add_artist(leg1)
    ax.legend(
        handles=year_handles, title='Biofuel price ramp-up year',
        loc='upper center', bbox_to_anchor=(0.5, -0.25),
        ncol=min(5, len(ramp_years_sorted)), frameon=True, fontsize=8,
    )
    
    ax.set_ylabel('Age of the vessel at scrapping (years)')
    ax.set_xlabel('')
    ax.grid(alpha=0.3)
    fig.subplots_adjust(bottom=0.35)
    
    plt.show()

## Fleet composition

In [ ]:
# Fleet composition breakdown by scenario (share of total dwt)
n_scenarios = len(scenarios_sub)

# Define engine groups with specific colors for better readability
color_map = {
    # Conventional group - blue shades
    'DF methanol': "#2E00FDFF", 
    'DF methanol + wind': "#8AD4FF", 
    
    # Ammonia group - red shades 
    'DF LNG': "#700000",  
    'DF LNG + CCS': "#FF0000", 
    'DF LNG + wind': "#FF6969", 
    
    # Methanol group - green shades 
    'DF ammonia': "#008100",  
    'DF ammonia + wind': "#00C400",  
    
    # LNG group - yellow/orange shades
    'conventional + CCS': '#FFA500',
    'conventional':  "#AC7000",
    'conventional + wind':"#FFEE00",
}

# Add any missing engines with default colors
all_engines = df_fleet['engine'].dropna().unique()
default_palette = plt.cm.tab20(np.linspace(0, 1, len(all_engines)))

for i, engine in enumerate(all_engines):
    if engine not in color_map:
        color_map[engine] = default_palette[i]

# Create subplots - arrange in rows (e.g., 2 rows for multiple scenarios)
n_cols = min(4, n_scenarios)
n_rows = (n_scenarios + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows + 1.5))

# Flatten axes array for easier iteration
if n_scenarios == 1:
    axes = [axes]
else:
    axes = axes.flatten() if isinstance(axes, np.ndarray) else [axes]

for idx, scenario in enumerate(scenarios_sub):
    ax = axes[idx]
    
    # Aggregate by year and engine
    df_scenario_fuel = df_fleet[
        (df_fleet['scenario_name'] == scenario)
    ].groupby(['game_year', 'engine'])['dwt_cohort'].sum().unstack(fill_value=0)

    # Convert to share of total dwt (%)
    df_scenario_share = df_scenario_fuel.div(df_scenario_fuel.sum(axis=1), axis=0).fillna(0) * 100
    
    # Create color list in the same order as columns
    colors = [color_map.get(col, 'gray') for col in df_scenario_share.columns]
    
    # Plot stacked area chart with grouped colors (no legend)
    df_scenario_share.plot(
        kind='bar',
        stacked=True,
        ax=ax,
        legend=False,
        lw=0,
        color=colors
    )
    
    ax.set_title(format_scenario_base_label(scenario), fontweight='bold')
    ax.set_ylabel('Share of fleet deadweight (%)')
    ax.set_xlabel('')
    ax.grid(axis='y', alpha=0.3)

# Hide unused subplots
for idx in range(n_scenarios, len(axes)):
    axes[idx].set_visible(False)

# Add figure-level title with ramp-up years
fig.suptitle(build_ramp_up_years_title(scenarios_sub), fontsize=12, fontweight='bold', y=0.98)

# Create common legend based on color_map below all graphs
legend_handles = [
    Patch(facecolor=color, edgecolor='black', linewidth=0.5, label=engine)
    for engine, color in color_map.items()
]

fig.legend(
    handles=legend_handles,
    loc='lower center',
    bbox_to_anchor=(0.5, -0.1),
    ncol=min(4, len(color_map)),
    fontsize=9,
    title='Engine',
    frameon=True
)

plt.tight_layout(rect=[0, 0.05, 1, 0.96])
plt.show()

if SAVE_OUTPUTS:
    # Save figure to figures folder
    fig.savefig(figures_dir / f"fleet_composition_v{version_nb}_round_{round_nb}.png", dpi=300, bbox_inches='tight')

    # Export combined scenario composition data as percentages
    df_all_scenarios_fuel = df_fleet.groupby(['scenario_name', 'game_year', 'engine'])['dwt_cohort'].sum().unstack(fill_value=0)
    df_all_scenarios_fuel = df_all_scenarios_fuel.div(df_all_scenarios_fuel.sum(axis=1), axis=0).fillna(0) * 100
    df_all_scenarios_fuel = df_all_scenarios_fuel.reset_index()

    export_path = output_csv_dir / f"fleet_composition_v{version_nb}_round_{round_nb}.csv"
    df_all_scenarios_fuel.to_csv(export_path, index=False)


In [ ]:
# Retrofit activity by engine type and scenario (% of fleet DWT)
# Filter for retrofit actions only
df_retrofits = df_fleet[df_fleet['action'].str.contains('retrofit', case=False, na=False)].copy()
n_scenarios = len(scenarios_sub)

# Add any missing engines with default colors
all_engines = df_retrofits['engine'].dropna().unique()
default_palette = plt.cm.tab20(np.linspace(0, 1, len(all_engines)))

for i, engine in enumerate(all_engines):
    if engine not in color_map:
        color_map[engine] = default_palette[i]

# Create subplots
n_cols = min(3, n_scenarios)
n_rows = (n_scenarios + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows + 1.5))

# Flatten axes array for easier iteration
if n_scenarios == 1:
    axes = [axes]
else:
    axes = axes.flatten() if isinstance(axes, np.ndarray) else [axes]

for idx, scenario in enumerate(scenarios_sub):
    ax = axes[idx]
    
    # Aggregate by year and engine for retrofits only
    df_scenario_retro = df_retrofits[
        (df_retrofits['scenario_name'] == scenario)
    ].groupby(['game_year', 'engine'])['dwt_cohort'].sum().unstack(fill_value=0)
    
    if df_scenario_retro.empty:
        ax.text(0.5, 0.5, 'No retrofits', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{scenario}', fontweight='bold')
        ax.set_xlabel('')
        continue
    
    # Get total fleet DWT for each year in this scenario
    total_fleet_dwt = df_fleet[
        df_fleet['scenario_name'] == scenario
    ].groupby('game_year')['dwt_cohort'].sum()
    
    # Convert to percentage of total fleet DWT
    df_scenario_retro_pct = df_scenario_retro.div(total_fleet_dwt, axis=0).fillna(0) * 100
    
    # Create color list in the same order as columns
    colors = [color_map.get(col, 'gray') for col in df_scenario_retro_pct.columns]
    
    # Plot stacked bar chart with percentage values
    df_scenario_retro_pct.plot(
        kind='bar',
        stacked=True,
        ax=ax,
        legend=False,
        color=colors,
        width=0.8
    )
    
    ax.set_title(f'{scenario}', fontweight='bold')
    ax.set_ylabel('Retrofitted dwt as share of fleet (%)')
    ax.set_ylim(0,40)
    ax.set_xlabel('')
    ax.grid(axis='y', alpha=0.3)

# Hide unused subplots
for idx in range(n_scenarios, len(axes)):
    axes[idx].set_visible(False)

# Create common legend
legend_handles = [
    Patch(facecolor=color, edgecolor='black', linewidth=0.5, label=engine)
    for engine, color in color_map.items()
]

fig.legend(
    handles=legend_handles,
    loc='lower center',
    bbox_to_anchor=(0.5, -0.1),
    ncol=min(4, len(color_map)),
    fontsize=9,
    title='Retrofit to Engine',
    frameon=True
)

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

if SAVE_OUTPUTS:
    fig.savefig(figures_dir / f"retrofits_by_engine_v{version_nb}_round_{round_nb}.png", dpi=300, bbox_inches='tight')
    export_path = output_csv_dir / f"retrofits_by_engine_v{version_nb}_round_{round_nb}.csv"
    df_retrofits.to_csv(export_path, index=False)


# df_retrofits is only used within this cell; safe to free.    
if CLEAN:
    del df_retrofits
    gc.collect()


## Fuel mix

In [ ]:

# Fuel composition breakdown by scenario (share of total fuel consumption)
df_fuel_long_sub = df_fuel_long.copy()


# Define fuel groups with specific colors for better readability
# Conventional/LSFO - blue shades
# LNG - yellow/orange shades
# Ammonia - red shades
# Methanol - green shades

fuel_color_map = {
    # Methanol group - blue shades
    'bio methanol/bio-ethanol': "#0019FD", 
    'e-methanol': "#0077FF", 
    
    # Fuel oil - yellow/orange shades
    'fuel oil': "#754C00",  
    'e-MGO': '#FFD700',  
    'bio-MGO': '#FF8C00',  

    # LNG group - red shades    
    'LNG': "#9C0000",  
    'bio-LNG': "#FF0000", 
    'e-LNG': "#FF6969", 
    
    # ammonia group - green shades
    # 'e-/blue ammonia': "#008100", 
    'e-ammonia': "#04FF04", 
    'blue ammonia': "#008100", 

}

# Filter for all fuels in dataset
fuels_ordered = [fuel for fuel, _ in fuel_color_map.items()]

n_scenarios = len(scenarios_sub)

# Create subplots - arrange in rows
n_cols = min(4, n_scenarios)
n_rows = (n_scenarios + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows + 1.5))

# Flatten axes array for easier iteration
if n_scenarios == 1:
    axes = [axes]
else:
    axes = axes.flatten() if isinstance(axes, np.ndarray) else [axes]

for idx, scenario in enumerate(scenarios_sub):
    ax = axes[idx]
    
    # Aggregate by year and fuel
    df_scenario_fuel = (
        df_fuel_long_sub[
            (df_fuel_long_sub['scenario_name'] == scenario)
            ]
        .groupby(['estimation_year', 'fuel'])['fuel_consumption_mj_year_cohort_excl_wind']
        .sum().unstack(fill_value=0) 
        *pow(10,-12)
    )
    fuels_ordered_sub = [c for c in fuels_ordered if c in df_scenario_fuel.columns]
    df_scenario_fuel = df_scenario_fuel[fuels_ordered_sub]
    
    # Create color list in the same order as columns
    colors = [fuel_color_map.get(col, '#C0C0C0') for col in df_scenario_fuel.columns]
    
    # Plot stacked area chart with grouped colors (no legend)
    df_scenario_fuel.plot(
        kind='bar',
        stacked=True,
        ax=ax,
        legend=False,
        lw=0,
        color=colors
    )
    
    ax.set_title(format_scenario_base_label(scenario), fontweight='bold')
    ax.set_ylabel('Fuel consumption (EJ)')
    ax.set_xlabel('')
    ax.grid(axis='y', alpha=0.3)

# Hide unused subplots
for idx in range(n_scenarios, len(axes)):
    axes[idx].set_visible(False)

# Add figure-level title with ramp-up years
fig.suptitle(build_ramp_up_years_title(scenarios_sub), fontsize=12, fontweight='bold', y=0.98)

# Create common legend based on fuel_color_map below all graphs
legend_handles = [
    Patch(facecolor=color, edgecolor='black', linewidth=0.5, label=fuel)
    for fuel, color in fuel_color_map.items()
]

fig.legend(
    handles=legend_handles,
    loc='lower center',
    bbox_to_anchor=(0.5, -0.1),
    ncol=min(4, len(fuel_color_map)),
    fontsize=9,
    title='Fuel',
    frameon=True
)

plt.tight_layout(rect=[0, 0.05, 1, 0.96])
plt.show()

if SAVE_OUTPUTS:
    fig.savefig(figures_dir / f"fuel_composition_v{version_nb}_round_{round_nb}.png", dpi=300, bbox_inches='tight')

    # Export combined scenario fuel composition data as percentages
    df_all_scenarios_fuel_cons = (
        df_fuel_long_sub
        .groupby(['scenario_name', 'estimation_year', 'fuel'])['fuel_consumption_mj_year_cohort_excl_wind']
        .sum()
        .unstack(fill_value=0)
        *pow(10,-12)
    )
    df_all_scenarios_fuel_cons = df_all_scenarios_fuel_cons.reset_index()

    export_path = output_csv_dir / f"fuel_composition_v{version_nb}_round_{round_nb}.csv"
    df_all_scenarios_fuel_cons.to_csv(export_path, index=False)


# df_fuel_long_sub is a full copy of df_fuel_long only used within this cell; safe to free.    
if CLEAN:
    del df_fuel_long_sub
    gc.collect()

In [ ]:
# E-fuel share in the fuel mix — color = ramp-up year, linestyle = base scenario
fig, ax = plt.subplots(figsize=(8, 6))

# Treat fuels whose labels start with e- or e-/ as e-fuels
efuel_pattern = r'^e[-/]'
e_fuel_share = (
    df_fuel_long[
        df_fuel_long['fuel'].str.contains(efuel_pattern, case=False, na=False, regex=True)
    ]
    .groupby(['estimation_year', 'scenario_name'])['fuel_consumption_mj_year_cohort_excl_wind']
    .sum()
    .rename('e_fuel_consumption_mj_year_cohort')
    .reset_index()
    .merge(
        df_fuel_long.groupby(['estimation_year', 'scenario_name'])['fuel_consumption_mj_year_cohort_excl_wind']
        .sum()
        .rename('total_fuel_consumption_mj_year_cohort')
        .reset_index(),
        on=['estimation_year', 'scenario_name'],
        how='left',
    )
)
e_fuel_share = e_fuel_share[e_fuel_share.scenario_name.isin(scenarios)]

assert len(e_fuel_share) > 0, 'No e-fuel rows found to plot e-fuel share'
assert e_fuel_share['total_fuel_consumption_mj_year_cohort'].gt(0).all(), (
    'Total fuel consumption must be positive for all scenario-year rows when computing e-fuel share'
 )
e_fuel_share['e_fuel_share'] = (
    e_fuel_share['e_fuel_consumption_mj_year_cohort']
    / e_fuel_share['total_fuel_consumption_mj_year_cohort']
    * 100
 )

# Parse scenario_name → base scenario + ramp-up year
e_fuel_share[['base_scenario', 'ramp_year']] = e_fuel_share['scenario_name'].apply(
    lambda x: pd.Series(split_scenario_base_and_ramp_year(x))
)
e_fuel_share['base_scenario'] = e_fuel_share['base_scenario'].apply(format_scenario_base_label)

# Assign colours by ramp-up year and linestyles by base scenario
ramp_years_sorted = sorted(e_fuel_share['ramp_year'].unique())
base_scenarios = sorted(e_fuel_share['base_scenario'].unique())

cmap = plt.cm.get_cmap('tab10', len(ramp_years_sorted))
year_colors = {yr: cmap(i) for i, yr in enumerate(ramp_years_sorted)}
base_styles = {bs: line_styles[i % len(line_styles)] for i, bs in enumerate(base_scenarios)}

for (bs, yr), grp in e_fuel_share.groupby(['base_scenario', 'ramp_year']):
    grp_sorted = grp.sort_values('estimation_year')
    ax.plot(
        grp_sorted['estimation_year'], grp_sorted['e_fuel_share'],
        color=year_colors[yr], linestyle=base_styles[bs], linewidth=2,
    )

# Build a two-part legend: one for years (color), one for base scenarios (linestyle)
year_handles = [Line2D([], [], color=year_colors[yr], lw=2, label=str(yr)) for yr in ramp_years_sorted]
style_handles = [Line2D([], [], color='grey', lw=2, linestyle=base_styles[bs], label=bs)
                 for bs in base_scenarios]

leg1 = ax.legend(
    handles=style_handles, title='Policy scenario',
    loc='upper center', bbox_to_anchor=(0.5, -0.1),
    ncol=min(3, len(base_scenarios)), frameon=True, fontsize=9,
 )
ax.add_artist(leg1)
# ax.legend(
#     handles=year_handles, title='Biofuel price ramp-up year',
#     loc='upper center', bbox_to_anchor=(0.5, -0.3),
#     ncol=min(5, len(ramp_years_sorted)), frameon=True, fontsize=8,
# )

ax.set_ylabel('E-fuels as share of fuel mix (%)')
ax.set_xlabel('')
ax.set_ylim(0, 20)
ax.grid(alpha=0.3)
# fig.subplots_adjust(bottom=0.35)

plt.show()
if SAVE_OUTPUTS:
    fig.savefig(figures_dir / f"e_fuel_share_v{version_nb}_round_{round_nb}.png", dpi=300, bbox_inches='tight')
    export_path = output_csv_dir / f"e_fuel_share_v{version_nb}_round_{round_nb}.csv"
    e_fuel_share.to_csv(export_path, index=False)

In [ ]:
# Electricity requirements for e-fuel production — color = ramp-up year, linestyle = base scenario
fig, ax = plt.subplots(figsize=(8, 6))

# Conversion efficiency: MJ fuel / MJ electricity
# source: https://www.concawe.eu/wp-content/uploads/Rpt_24-4-1.pdf
conversion_efficiency = {
    'e-ammonia': 0.59,
    'e-LNG': 0.52,
    'e-methanol': 0.48,
    'e-MGO': 0.42,
}

# Filter for e-fuels and calculate electricity requirements
efuel_pattern = r'^e[-/]'
e_fuel_data = df_fuel_long[
    df_fuel_long['fuel'].str.contains(efuel_pattern, case=False, na=False, regex=True)
].copy()

# Remove rows where we couldn't match to a known e-fuel
e_fuel_data = e_fuel_data[e_fuel_data['fuel'].notna()]

# Calculate electricity requirements: (MJ fuel/year) / (MJ fuel/MJ electricity) / 3.6 (MJ/kWh) = kWh/year
e_fuel_data['electricity_kwh_year'] = e_fuel_data.apply(
    lambda row: (
        row['fuel_consumption_mj_year_cohort_excl_wind'] 
        / conversion_efficiency[row['fuel']]
        / 3.6
    ),
    axis=1
)

# Aggregate by scenario and year
electricity_req = (
    e_fuel_data
    .groupby(['estimation_year', 'scenario_name'])['electricity_kwh_year']
    .sum()
    .reset_index()
)

electricity_req = electricity_req[electricity_req.scenario_name.isin(scenarios)]

assert len(electricity_req) > 0, 'No e-fuel electricity requirement data found'

# Parse scenario_name → base scenario + ramp-up year
electricity_req[['base_scenario', 'ramp_year']] = electricity_req['scenario_name'].apply(
    lambda x: pd.Series(split_scenario_base_and_ramp_year(x))
)
electricity_req['base_scenario'] = electricity_req['base_scenario'].apply(format_scenario_base_label)

# Assign colours by ramp-up year and linestyles by base scenario
ramp_years_sorted = sorted(electricity_req['ramp_year'].unique())
base_scenarios = sorted(electricity_req['base_scenario'].unique())

cmap = plt.cm.get_cmap('tab10', len(ramp_years_sorted))
year_colors = {yr: cmap(i) for i, yr in enumerate(ramp_years_sorted)}
base_styles = {bs: line_styles[i % len(line_styles)] for i, bs in enumerate(base_scenarios)}

for (bs, yr), grp in electricity_req.groupby(['base_scenario', 'ramp_year']):
    grp_sorted = grp.sort_values('estimation_year')
    ax.plot(
        grp_sorted['estimation_year'], grp_sorted['electricity_kwh_year'] / 1e9,  # Convert to TWh
        color=year_colors[yr], linestyle=base_styles[bs], linewidth=2,
    )

# Build a two-part legend: one for years (color), one for base scenarios (linestyle)
year_handles = [Line2D([], [], color=year_colors[yr], lw=2, label=str(yr)) for yr in ramp_years_sorted]
style_handles = [Line2D([], [], color='grey', lw=2, linestyle=base_styles[bs], label=bs)
                 for bs in base_scenarios]

leg1 = ax.legend(
    handles=style_handles, title='Policy scenario',
    loc='upper center', bbox_to_anchor=(0.5, -0.1),
    ncol=min(3, len(base_scenarios)), frameon=True, fontsize=9,
 )
ax.add_artist(leg1)
ax.legend(
    handles=year_handles, title='Biofuel price ramp-up year',
    loc='upper center', bbox_to_anchor=(0.5, -0.3),
    ncol=min(5, len(ramp_years_sorted)), frameon=True, fontsize=8,
)

ax.set_ylabel('Electricity for e-fuel production (TWh/year)')
ax.set_xlabel('')
ax.grid(alpha=0.3)
fig.subplots_adjust(bottom=0.35)

plt.show()
if SAVE_OUTPUTS:
    fig.savefig(figures_dir / f"e_fuel_electricity_v{version_nb}_round_{round_nb}.png", dpi=300, bbox_inches='tight')
    export_path = output_csv_dir / f"e_fuel_electricity_v{version_nb}_round_{round_nb}.csv"
    electricity_req.to_csv(export_path, index=False)

In [ ]:
df_fuel_long_merged = df_fuel_long.copy()
# mask = df_fuel_long_merged[df_fuel_long_merged.fuel.isin(["e-ammonia","blue ammonia"])].index
# df_fuel_long_merged.loc[mask, 'fuel'] = "e-/blue ammonia"

fuel_infra = (
    df_fuel_long_merged
    .groupby(['scenario_name', 'estimation_year', 'fuel'])['fuel_consumption_mj_year_cohort_excl_wind']
    .sum()
    .unstack(fill_value=0)   # columns = fuel
    .sort_index()
)

step = 3

# enforce monotonic infrastructure by scenario
for scenario, year in fuel_infra.index:
    prev_key = (scenario, year - step)
    curr_key = (scenario, year)
    if prev_key in fuel_infra.index:
        fuel_infra.loc[curr_key] = fuel_infra.loc[[curr_key, prev_key]].max(axis=0)

growth_fuel_infra = fuel_infra.copy()

# growth every 3 years by scenario
for scenario, year in fuel_infra.index:
    prev_key = (scenario, year - step)
    curr_key = (scenario, year)
    if prev_key in fuel_infra.index:
        growth_fuel_infra.loc[curr_key] = (fuel_infra.loc[curr_key] - fuel_infra.loc[prev_key])/step
    else:
        growth_fuel_infra.loc[curr_key] = fuel_infra.loc[curr_key]/step

growth_fuel_infra

In [ ]:
# harmonize labels with your previous fuel grouping
growth_plot = growth_fuel_infra.copy()*pow(10,-12)

n_scenarios = len(scenarios_sub)

n_cols = min(3, n_scenarios)
n_rows = (n_scenarios + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows + 1.5))

if n_scenarios == 1:
    axes = [axes]
else:
    axes = axes.flatten() if isinstance(axes, np.ndarray) else [axes]

# ordered fuels: first those in color map, then any extras
fuels_present = list(growth_plot.columns)
fuels_ordered = [f for f in fuel_color_map if f in fuels_present] + [f for f in fuels_present if f not in fuel_color_map]

for idx, scenario in enumerate(scenarios_sub):
    ax = axes[idx]
    df_s = growth_plot.xs(scenario, level='scenario_name').copy()
    df_s = df_s.reindex(columns=fuels_ordered, fill_value=0).sort_index()

    colors = [fuel_color_map.get(col, '#C0C0C0') for col in df_s.columns]
    df_s.loc[2028:].plot(
        kind='bar',
        stacked=True,
        ax=ax,
        legend=False,
        lw=0,
        color=colors
    )

    ax.set_title(f'{scenario}', fontweight='bold')
    ax.set_ylabel(f'Additional annual fuel production capacity\n(EJ/year, over {step} years)')
    ax.set_xlabel('')
    ax.set_ylim(0,2)
    ax.grid(axis='y', alpha=0.3)

# hide unused subplots
for idx in range(n_scenarios, len(axes)):
    axes[idx].set_visible(False)

# common legend
legend_handles = [
    Patch(facecolor=fuel_color_map.get(f, '#C0C0C0'), edgecolor='black', linewidth=0.5, label=f)
    for f in fuels_ordered
]
fig.legend(
    handles=legend_handles,
    loc='lower center',
    bbox_to_anchor=(0.5, -0.05),
    ncol=min(4, len(fuels_ordered)),
    fontsize=9,
    title='Fuel',
    frameon=True
)

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

if SAVE_OUTPUTS:
    fig.savefig(figures_dir / f"additional_fuel_prod_v{version_nb}_round_{round_nb}.png", dpi=300, bbox_inches='tight')

    export_path = output_csv_dir / f"additional_fuel_prod_v{version_nb}_round_{round_nb}.csv"
    growth_plot.to_csv(export_path, index=False)

## Emissions

In [ ]:
# reduction in fleet CO2e em compared to 2008 baseline
imo_strat = {
    'year': [2030, 2040, 2050],
    'base': [-0.2, -0.7, -1],
    'strive': [-0.3, -0.8, -1],
}


# the scope of the current fleet and of the strategy is different
# this fleet covers ~85% of international shipping emissions in 2018 (it excludes cruise, ferries and smaller segments)
# however, the emissions included here cover both international and domestic shipping emissions, whereas the IMO strategy only covers international emissions (however, the sample is made of segments which spend most of their time doing interntional voyages)
# rather than calcualting the actual CO2e of this fleet, we assume that :
# 1/ emissins are roughly constant between 2008 and 2025 (the starting year) based on the DNV CIA's figure D-1
# 2/ emissions need to decrease in the same proportion btw this sample and the international shipping fleet sample

# Emissions by year — color = ramp-up year, linestyle = base scenario
fig, ax_norm = plt.subplots(figsize=(8, 6))

to_plot = (
    df_fuels_filtered_annual[df_fuels_filtered_annual.scenario_name.isin(scenarios)]
    .groupby(['estimation_year', 'scenario_name'])
    .tonne_co2e
    .sum()
    .unstack()
    * pow(10, -6)
)

# Parse scenario_name -> base scenario + ramp-up year
emission_df = to_plot.reset_index().melt(
    id_vars='estimation_year',
    var_name='scenario_name',
    value_name='million_tonne_co2e',
)
emission_df = emission_df[emission_df['million_tonne_co2e'].notna()].copy()
emission_df[['base_scenario', 'ramp_year']] = emission_df['scenario_name'].apply(
    lambda x: pd.Series(split_scenario_base_and_ramp_year(x))
)
emission_df['base_scenario'] = emission_df['base_scenario'].apply(format_scenario_base_label)

assert len(emission_df) > 0, 'No emission data found to plot'

# Assign colours by ramp-up year and linestyles by base scenario
ramp_years_sorted = sorted(emission_df['ramp_year'].unique())
base_scenarios = sorted(emission_df['base_scenario'].unique())

cmap = plt.cm.get_cmap('tab10', len(ramp_years_sorted))
year_colors = {yr: cmap(i) for i, yr in enumerate(ramp_years_sorted)}
base_styles = {bs: line_styles[i % len(line_styles)] for i, bs in enumerate(base_scenarios)}

for (bs, yr), grp in emission_df.groupby(['base_scenario', 'ramp_year']):
    grp_sorted = grp.sort_values('estimation_year')
    ax_norm.plot(
        grp_sorted['estimation_year'], grp_sorted['million_tonne_co2e'],
        color=year_colors[yr], linestyle=base_styles[bs], linewidth=2,
    )

# --- IMO reference trajectories (base and strive), interpolated from starting_year onward ---
years = np.sort(emission_df['estimation_year'].unique().astype(int))
years = years[years >= starting_year]
assert len(years) > 0, 'No years >= starting_year found in emissions data'

baseline = (
    df_fuels_filtered_annual[
        df_fuels_filtered_annual['estimation_year'] == df_fuels_filtered_annual['estimation_year'].min()
    ]
    .groupby('scenario_name')['tonne_co2e']
    .sum()
    .mean()
    * pow(10, -6)
)
assert baseline > 0, 'Could not compute a positive starting_year emissions baseline for IMO trajectories'

interp_years = np.array([starting_year] + imo_strat['year'], dtype=int)
interp_base = np.array([0.0] + imo_strat['base'], dtype=float)
interp_strive = np.array([0.0] + imo_strat['strive'], dtype=float)

base_reduction_factor = np.interp(years, interp_years, interp_base, left=0.0, right=interp_base[-1])
strive_reduction_factor = np.interp(years, interp_years, interp_strive, left=0.0, right=interp_strive[-1])

annual_base_path = np.maximum(0.0, baseline * (1.0 + base_reduction_factor))
annual_strive_path = np.maximum(0.0, baseline * (1.0 + strive_reduction_factor))

ax_norm.plot(years, annual_base_path, color='grey', linewidth=1)
ax_norm.plot(years, annual_strive_path, color='grey', linewidth=2)

# Build legends: ramp-up years (color), base scenarios (linestyle), IMO references (grey lines)
year_handles = [Line2D([], [], color=year_colors[yr], lw=2, label=str(yr)) for yr in ramp_years_sorted]
style_handles = [Line2D([], [], color='grey', lw=2, linestyle=base_styles[bs], label=bs)
                 for bs in base_scenarios]
imo_handles = [
    Line2D([], [], color='grey', lw=1, label='base trajectory'),
    Line2D([], [], color='grey', lw=2, label='strive trajectory'),
]

leg1 = ax_norm.legend(
    handles=style_handles, title='Policy scenario',
    loc='upper center', bbox_to_anchor=(0.5, -0.1),
    ncol=min(2, len(base_scenarios)), frameon=True, fontsize=9, handlelength=4,
)
ax_norm.add_artist(leg1)
leg2 = ax_norm.legend(
    handles=year_handles, title='Biofuel price ramp-up year',
    loc='upper center', bbox_to_anchor=(0.5, -0.3),
    ncol=min(5, len(ramp_years_sorted)), frameon=True, fontsize=8,
)
ax_norm.add_artist(leg2)
ax_norm.legend(
    handles=imo_handles, title='IMO revised Strategy',
    loc='upper center', bbox_to_anchor=(0.5, -0.40),
    ncol=2, frameon=True, fontsize=8,
)

ax_norm.set_ylabel('Annual emissions (million tonne CO2e)')
ax_norm.set_xlabel('')
ax_norm.grid(alpha=0.3)
fig.subplots_adjust(bottom=0.60)
plt.tight_layout()
plt.show()

if SAVE_OUTPUTS:
    # Save figure to figures folder
    fig.savefig(figures_dir / f"emissions_v{version_nb}_round_{round_nb}.png", dpi=300, bbox_inches='tight')


In [ ]:
# Cumulative emissions by year — color = ramp-up year, linestyle = base scenario
fig, ax = plt.subplots(figsize=(8, 6))

cumulative_emissions = (
    df_fuels_filtered_annual[df_fuels_filtered_annual.scenario_name.isin(scenarios)]
    .groupby(['estimation_year', 'scenario_name'])['tonne_co2e']
    .sum()
    .unstack()
    .fillna(0)
    * pow(10, -6)
)
cumulative_emissions = cumulative_emissions[cumulative_emissions.index >= starting_year].cumsum()

# Parse scenario_name -> base scenario + ramp-up year
emission_df = cumulative_emissions.reset_index().melt(
    id_vars='estimation_year',
    var_name='scenario_name',
    value_name='cumulative_million_tonne_co2e',
)
emission_df = emission_df[emission_df['cumulative_million_tonne_co2e'].notna()].copy()
emission_df[['base_scenario', 'ramp_year']] = emission_df['scenario_name'].apply(
    lambda x: pd.Series(split_scenario_base_and_ramp_year(x))
)
emission_df['base_scenario'] = emission_df['base_scenario'].apply(format_scenario_base_label)

assert len(emission_df) > 0, 'No cumulative emission data found to plot'

# Assign colours by ramp-up year and linestyles by base scenario
ramp_years_sorted = sorted(emission_df['ramp_year'].unique())
base_scenarios = sorted(emission_df['base_scenario'].unique())

cmap = plt.get_cmap('tab10', len(ramp_years_sorted))
year_colors = {yr: cmap(i) for i, yr in enumerate(ramp_years_sorted)}
base_styles = {bs: line_styles[i % len(line_styles)] for i, bs in enumerate(base_scenarios)}

for (bs, yr), grp in emission_df.groupby(['base_scenario', 'ramp_year']):
    grp_sorted = grp.sort_values('estimation_year')
    ax.plot(
        grp_sorted['estimation_year'], grp_sorted['cumulative_million_tonne_co2e'],
        color=year_colors[yr], linestyle=base_styles[bs], linewidth=2,
    )

# --- IMO reference trajectories (base and strive), from annual_base_path / annual_strive_path
# computed in the previous cell using df_fuels_filtered_annual ---
years = cumulative_emissions.index.to_numpy(dtype=int)
years = np.sort(years[years >= starting_year])
assert len(years) > 0, 'No years >= starting_year found in cumulative emissions index'

baseline = (
    df_fuels_filtered_annual[
        df_fuels_filtered_annual['estimation_year'] == df_fuels_filtered_annual['estimation_year'].min()
    ]
    .groupby('scenario_name')['tonne_co2e']
    .sum()
    .mean()
    * pow(10, -6)
)
assert baseline > 0, 'Could not compute a positive starting_year emissions baseline for IMO trajectories'

cum_base_path = np.cumsum(annual_base_path)
cum_strive_path = np.cumsum(annual_strive_path)

ax.plot(years, cum_base_path, color='black', linewidth=1)
ax.plot(years, cum_strive_path, color='black', linewidth=2)

# Build legends: ramp-up years (color), base scenarios (linestyle), IMO references (black lines)
year_handles = [Line2D([], [], color=year_colors[yr], lw=2, label=str(yr)) for yr in ramp_years_sorted]
style_handles = [Line2D([], [], color='grey', lw=2, linestyle=base_styles[bs], label=bs)
                 for bs in base_scenarios]
imo_handles = [
    Line2D([], [], color='black', lw=1, label='base trajectory'),
    Line2D([], [], color='black', lw=2, label='strive trajectory'),
]

leg1 = ax.legend(
    handles=style_handles, title='Policy scenario',
    loc='upper center', bbox_to_anchor=(0.5, -0.1),
    ncol=min(2, len(base_scenarios)), frameon=True, fontsize=9, handlelength=4,
)
ax.add_artist(leg1)
leg2 = ax.legend(
    handles=year_handles, title='Biofuel price ramp-up year',
    loc='upper center', bbox_to_anchor=(0.5, -0.3),
    ncol=min(5, len(ramp_years_sorted)), frameon=True, fontsize=8,
)
ax.add_artist(leg2)
ax.legend(
    handles=imo_handles, title='IMO revised Strategy',
    loc='upper center', bbox_to_anchor=(0.5, -0.40),
    ncol=2, frameon=True, fontsize=8,
)

ax.set_ylabel('Cumulative (million tonne CO2e)')
ax.set_xlabel('')
ax.grid(alpha=0.3)
fig.subplots_adjust(bottom=0.60)
plt.tight_layout()
plt.show()

if SAVE_OUTPUTS:
    fig.savefig(figures_dir / f"cumulative_emissions_v{version_nb}_round_{round_nb}.png", dpi=300, bbox_inches='tight')
    export_path = output_csv_dir / f"cumulative_emissions_v{version_nb}_round_{round_nb}.csv"
    cumulative_emissions.to_csv(export_path)


In [ ]:
print("Cumulative emissions compared to the strive trajectory")
(
    cumulative_emissions.iloc[-1]
    -cum_strive_path[-1]
)/cum_strive_path[-1]

## Fund

In [ ]:
years_to_plot = range(starting_year,2051,year_step)
scenarios_sub2 = [x for x in scenarios_sub if x in df_fuels_filtered_annual.scenario_name.unique()]
# Built from the shared annual_fund_components_usd helper (Functions cell above), so this stays in
# sync with fund_after_reward_ever_negative and cumulative_fund_after_reward_for.
df_fund = pd.concat(
    [annual_fund_components_usd(s).assign(scenario_name=s) for s in scenarios],
    ignore_index=True,
)
df_fund[['revenue', 'fund_after_reward', 'reward_usd']] *= pow(10, -9)

years = sorted(years_to_plot)
n_scenarios = len(scenarios_sub2)

colors = plt.cm.tab10(np.linspace(0, 1, n_scenarios))
# Create clean labels for scenarios
scenarios_sub2_clean = [format_scenario_base_label(s) for s in scenarios_sub2]
scenario_color = dict(zip(scenarios_sub2_clean, colors))

bar_width = 0.8 / n_scenarios
x = np.arange(len(years_to_plot))
y_max = 20
y_min = 0

fig, ax = plt.subplots(figsize=(8, 6))

for i, (scenario, scenario_clean) in enumerate(zip(scenarios_sub2, scenarios_sub2_clean)):
    sub = df_fund[
        (df_fund['scenario_name'] == scenario)
        &(df_fund.estimation_year.isin(years_to_plot))
    ].set_index('estimation_year').reindex(years)
    x_pos = x + (i - n_scenarios / 2 + 0.5) * bar_width
    color = scenario_color[scenario_clean]

    revenue = sub['revenue'].fillna(0).values
    reward = sub['reward_usd'].fillna(0).values
    fund = sub['fund_after_reward'].fillna(0).values

    # Positive part: revenue as solid bar (full height = revenue)
    ax.bar(x_pos, revenue, width=bar_width,
           color=color, edgecolor='black', linewidth=0.4, label=scenario_clean)

    # Reward overlay: hatched, sits on top of fund_after_reward
    ax.bar(x_pos, reward, width=bar_width,
           bottom=0,
           color=color, edgecolor='black', linewidth=0.4, hatch='//',
           zorder=3)

    # Negative overhang: fund_after_reward below zero (reward exceeds revenue)
    neg_fund = np.where(fund < 0, fund, 0)
    ax.bar(x_pos, neg_fund, width=bar_width,
           color=color, edgecolor='black', linewidth=0.4, alpha=0.4,
           zorder=2)

    y_max = max(y_max, max(revenue)*1.1)
    y_min = min(y_min, min(neg_fund)*1.1)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xticks(x)
ax.set_xticklabels(years, rotation=45, ha='right')
ax.set_ylabel('Billion USD/year')
ax.set_xlabel('')
ax.set_ylim(y_min, y_max)

scenario_handles = [
    Patch(facecolor=scenario_color[s], edgecolor='black', linewidth=0.5, label=s)
    for s in scenarios_sub2_clean
]
style_handles = [
    Patch(facecolor='lightgray', edgecolor='black', linewidth=0.5,
          label='Revenue / fund after reward (solid)'),
    Patch(facecolor='lightgray', edgecolor='black', linewidth=0.5, hatch='//',
          label='Reward disbursed (hatched)'),
    Patch(facecolor='lightgray', edgecolor='black', linewidth=0.5, alpha=0.4,
          label='Fund deficit – reward exceeds revenue (faded, below zero)'),
]
ax.legend(
    handles=scenario_handles + style_handles,
    loc='upper left', bbox_to_anchor=(0, -0.15),
    borderaxespad=0.0, fontsize=9, ncol=2
)

# Add figure title showing ramp-up years
ax.set_title(build_ramp_up_years_title(scenarios_sub2), fontweight='bold')

# plt.tight_layout(rect=[0, 0, 0.78, 1])
plt.show()


if SAVE_OUTPUTS:
    # Save figure to figures folder
    fig.savefig(figures_dir / f"fund_v{version_nb}_round_{round_nb}.png", dpi=300, bbox_inches='tight')
    export_path = output_csv_dir / f"fund_v{version_nb}_round_{round_nb}.csv"
    df_fund[(df_fund.estimation_year.isin(years_to_plot))].to_csv(export_path, index=False)
df_fund.groupby('scenario_name').fund_after_reward.sum()

## Transport costs

In [ ]:
cost_cols = [
    'capex_annualized',
    'annual_opex_excl_fuel_policy',
    'stranded_value',
    'total_fuel_cost',
]
df_fleet_to_plot = df_fleet.copy()

# Exclude annualized capex beyond depreciation time to avoid double counting.
mask = df_fleet_to_plot[df_fleet_to_plot.game_year - df_fleet_to_plot.year_built > DEPRECIATION_TIME].index
df_fleet_to_plot.loc[mask, 'capex_annualized'] = 0

# scale the costs at the fleet level
all_cost_cols = cost_cols + (['net_policy_cost'] if 'net_policy_cost' not in cost_cols else [])
for c in all_cost_cols:
    df_fleet_to_plot[c] *= df_fleet_to_plot['real_ships_nb']
df_fleet_to_plot['transport_work_dwt_nm'] *= df_fleet_to_plot['real_ships_nb']

fig, ax = plt.subplots(figsize=(8, 6))

# Interpolate fleet costs to annual values before summing cumulatively
df_fleet_annual = interpolate_to_annual(
    df_fleet_to_plot
    .groupby(['scenario_name', 'game_year'])[cost_cols]
    .sum()
    .reset_index(),
    year_col='game_year',
    group_cols=['scenario_name'],
    value_cols=cost_cols,
)
cum_cost_usd = df_fleet_annual.groupby('scenario_name')[cost_cols].sum()

# Add policy separately to avoid SU market loop artifacts in ship-level costs.
cum_policy_cost = (
    df_fund.groupby(['scenario_name'])['fund_after_reward']
    .sum()
    * pow(10, 9)
)
check_len = len(cum_cost_usd)
cum_cost_usd = cum_cost_usd.reset_index().merge(
    cum_policy_cost.reset_index().rename(columns={'fund_after_reward': 'policy_cost'}),
    on=['scenario_name'],
    how='left',
).set_index('scenario_name').fillna(0)
assert check_len == len(cum_cost_usd)

# df_fleet_annual and cum_policy_cost are only intermediates used to build cum_cost_usd above.
if CLEAN:
    del df_fleet_annual, cum_policy_cost
    gc.collect()

# SCC term: reuse cumulative emissions and IMO base trajectory produced in the Emissions section.
assert 'cumulative_emissions' in locals(), "Run the Emissions cells first to create cumulative_emissions"
assert 'cum_strive_path' in locals(), "Run the Emissions cells first to create cum_strive_path"

model_years = np.sort(cumulative_emissions.index.to_numpy(dtype=int))
model_years = model_years[model_years >= baseline_year]
assert len(model_years) > 0, f"No years >= {baseline_year} available in cumulative_emissions"
assert len(cum_strive_path) == len(model_years), "Emissions cell outputs are out of sync; rerun Emissions cells"

imo_strive_cum_series = pd.Series(cum_strive_path, index=model_years)
scenario_cum = cumulative_emissions.reindex(index=model_years)
cum_delta_end_million_tco2e = scenario_cum.iloc[-1] - imo_strive_cum_series.iloc[-1]

# Signed SCC: above IMO strive target -> positive cost; below target -> negative cost.
scc_cost = cum_delta_end_million_tco2e * pow(10, 6) * scc_usd_p_tco2e

cum_cost_usd['Social cost of carbon'] = scc_cost.reindex(cum_cost_usd.index).fillna(0)

# Subtract BAU scenario costs from each scenario to show difference relative to BAU
cum_cost_usd_bau = cum_cost_usd.loc[bau_label]
net_cum_cost_usd = cum_cost_usd.sub(cum_cost_usd.loc[bau_label])

to_plot = net_cum_cost_usd * pow(10, -9)
to_plot = to_plot.loc[[x for x in scenarios_sub if x != bau_label]]
to_plot.plot(kind="bar", stacked=True, ax=ax)

# Clean x labels and move ramp-up years to title using reusable helper functions.
clean_labels = [format_scenario_base_label(name) for name in to_plot.index]
ax.set_xticks(np.arange(len(clean_labels)))
ax.set_xticklabels(clean_labels, rotation=45, ha='right')
ax.set_title(build_ramp_up_years_title(to_plot.index))

# Overlay net total to make mixed positive/negative stacks readable.
total_cost_billion = to_plot.sum(axis=1)
x_pos = np.arange(len(total_cost_billion))
ax.plot(x_pos, total_cost_billion.values, color='black', marker='o', linewidth=2.5, label='Total net cost')

handles, labels = ax.get_legend_handles_labels()
clean_labels = [" ".join(label.replace('planning_horizon', '').replace('_', ' ').split()) for label in labels]
ax.legend(handles, clean_labels, title='Cost component', loc='best')
ax.set_xlabel("")
ax.set_ylabel(f"Cumulative additional cost compared \nto BAU {baseline_year}-2050 (billion USD)")
plt.tight_layout()
plt.show()

print("cumulative addigional costs compared to bau: ")
print(total_cost_billion/(cum_cost_usd_bau.sum()*pow(10,-9)))

if SAVE_OUTPUTS:
    fig.savefig(figures_dir / f"cumulative_additional_costs_v{version_nb}_round_{round_nb}.png", dpi=300, bbox_inches='tight')
    export_path = output_csv_dir / f"cumulative_additional_costs_v{version_nb}_round_{round_nb}.csv"
    to_plot.to_csv(export_path, index=False)

In [ ]:
# Cost components by year - one subplot per scenario
n_scenarios = len(scenarios_sub)
n_cols = min(3, n_scenarios)
n_rows = int(np.ceil(n_scenarios / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows), squeeze=False, sharey = True)
axes = axes.flatten()

for idx, scenario in enumerate(scenarios_sub):
    ax = axes[idx]
    
    # Filter data for this scenario and aggregate by year
    df_scenario = (
        df_fleet_to_plot[df_fleet_to_plot.scenario_name == scenario]
        .groupby('game_year')[cost_cols]
        .sum()
    )
    
    # Add fund_after_reward from df_fund (convert from billions to match other costs)
    df_fund_scenario = (
        df_fund[df_fund.scenario_name == scenario]
        .set_index('estimation_year')['fund_after_reward']
        * pow(10, 9)  # Convert from billions
    )
    
    # Merge fund data with scenario data
    df_scenario['fund_after_reward'] = df_fund_scenario.reindex(df_scenario.index).fillna(0)
    
    # Plot stacked bar chart
    df_scenario.plot(stacked=True, kind="bar", ax=ax, legend=(idx == 0))
    
    # Set title using clean scenario label
    clean_label = format_scenario_base_label(scenario)
    ramp_year_match = re.search(r'ramp_up_(\d{4})', scenario)
    if ramp_year_match:
        ramp_year = ramp_year_match.group(1)
        ax.set_title(f"{clean_label}\nRamp-up year: {ramp_year}", fontweight='bold')
    else:
        ax.set_title(clean_label, fontweight='bold')
    
    ax.set_xlabel('')
    ax.set_ylabel('Annual cost (USD)')
    ax.tick_params(axis='x', rotation=45)

# Remove extra subplots if any
for idx in range(n_scenarios, len(axes)):
    fig.delaxes(axes[idx])

# Create a single legend for all subplots
if n_scenarios > 0:
    handles, labels = axes[0].get_legend_handles_labels()
    clean_labels = [" ".join(label.replace('_', ' ').split()) for label in labels]
    fig.legend(handles, clean_labels, title='Cost component', loc='upper center', 
               bbox_to_anchor=(0.5, 0.0), ncol=len(labels), frameon=True)
    # Remove individual subplot legends
    for ax in axes[:n_scenarios]:
        if ax.get_legend():
            ax.get_legend().remove()

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

In [ ]:
# Cumulative additional cost vs BAU by ramp-up year and policy scenario
# Calculate total net cost (sum across all cost components)
df_total_cost = net_cum_cost_usd.copy()
df_total_cost['total_net_cost'] = df_total_cost.sum(axis=1) * pow(10, -9)  # Convert to billions
df_total_cost = df_total_cost[['total_net_cost']].reset_index()

# Exclude BAU scenario
df_total_cost = df_total_cost[~df_total_cost['scenario_name'].str.contains("No IMO") & (df_total_cost.scenario_name.isin(scenarios))]

# Parse scenario_name to extract base scenario and ramp-up year
df_total_cost[['base_scenario', 'ramp_year']] = df_total_cost['scenario_name'].apply(
    lambda x: pd.Series(split_scenario_base_and_ramp_year(x))
)
df_total_cost['base_scenario'] = df_total_cost['base_scenario'].apply(format_scenario_base_label)

# Create line plot: x-axis = ramp-up year, lines = policy scenarios
fig, ax = plt.subplots(figsize=(8, 6))

# Define line styles for different policy scenarios
pivot_data = df_total_cost.groupby(['ramp_year','base_scenario']).total_net_cost.mean().unstack()

# Plot each policy scenario as a separate line with different linestyle
for idx, base_scenario in enumerate(pivot_data.columns):
    linestyle = line_styles[idx % len(line_styles)]
    ax.plot(
        pivot_data.index,
        pivot_data[base_scenario],
        linestyle=linestyle,
        linewidth=2.5,
        color='black',
        label=base_scenario,
        marker = "o"
    )

ax.set_xlabel('Biofuel price ramp-up year')
ax.set_ylabel(f"Cumulative additional cost compared \nto BAU {baseline_year}-2050 (billion USD)")
ax.legend(title='Policy scenario', loc='best', handlelength=5)
# ax.legend().remove()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Carbon abatement cost by ramp-up year and policy scenario
# Carbon abatement cost = (cumulative additional cost excluding social cost) / carbon saved.
# Uses the shared per-scenario helper (Utility functions cell above) instead of duplicating the
# net_cum_cost_usd / cumulative_emissions logic, so this stays in sync with the Sensitivity section.

bau_cum_emissions_mt = cumulative_emissions_mt_for(bau_label)
assert pd.notna(bau_cum_emissions_mt), "Could not compute BAU cumulative emissions"

df_abatement = pd.DataFrame({'scenario_name': [s for s in scenarios if s != bau_label]})
df_abatement['abatement_cost_usd_tco2e'] = df_abatement['scenario_name'].apply(abatement_cost_usd_tco2e_for)

# Parse scenario_name to extract base scenario and ramp-up year
df_abatement[['base_scenario', 'ramp_year']] = df_abatement['scenario_name'].apply(
    lambda x: pd.Series(split_scenario_base_and_ramp_year(x))
)
df_abatement['base_scenario'] = df_abatement['base_scenario'].apply(format_scenario_base_label)

# Create line plot: x-axis = ramp-up year, lines = policy scenarios
fig, ax = plt.subplots(figsize=(8, 6))

# Define line styles for different policy scenarios
pivot_data = df_abatement.groupby(['ramp_year', 'base_scenario']).abatement_cost_usd_tco2e.mean().unstack()

# Plot each policy scenario as a separate line with different linestyle
for idx, base_scenario in enumerate(pivot_data.columns):
    linestyle = line_styles[idx % len(line_styles)]
    ax.plot(
        pivot_data.index,
        pivot_data[base_scenario],
        linestyle=linestyle,
        linewidth=2.5,
        color='black',
        label=base_scenario
    )

ax.set_xlabel('Biofuel price ramp-up year')
ax.set_ylabel('USD per tCO2e saved')
ax.legend(title='Policy scenario', loc='best', handlelength=5, fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Print summary statistics
print("\nCarbon abatement cost summary (USD/tCO2e):")
print(df_abatement.groupby('base_scenario')['abatement_cost_usd_tco2e'].describe()[['mean', 'min', 'max']])

if SAVE_OUTPUTS:
    # Save figure to figures folder
    export_path = output_csv_dir / f"carbon_abatement_cost_v{version_nb}_round_{round_nb}.csv"
    pivot_data.to_csv(export_path)
    fig.savefig(figures_dir / f"carbon_abatement_cost_v{version_nb}_round_{round_nb}.png", dpi=300, bbox_inches='tight')


In [ ]:
cost_cols_incl_net_policy_cost = [
    'annual_opex_excl_fuel_policy', 
    'capex_annualized', 
    'total_fuel_cost', 
    'stranded_value',
    'net_policy_cost',
]

df_fleet_to_plot['transport_cost'] =  df_fleet_to_plot[cost_cols_incl_net_policy_cost].sum(axis = 1)
df_fleet_to_plot['transport_cost_usd_dwt_nm'] = df_fleet_to_plot['transport_cost'] / df_fleet_to_plot['transport_work_dwt_nm']

imo_type = "Container"
size_bin = 3

# Filter for the same ship type and size as cell 77
df_filtered = df_fleet_to_plot[
    (df_fleet_to_plot['imo_type'] == imo_type)
    & (df_fleet_to_plot['size_id'] == size_bin)
    &(df_fleet_to_plot.scenario_name.isin(scenarios))
    &(
        (df_fleet_to_plot.scenario_name.str.contains(str(starting_year + 9)))
        # |(df_fleet_to_plot.scenario_name.str.contains(str(starting_year + 15)))
    )
]

transport_mean = (
    df_filtered
    .groupby(['game_year', 'scenario_name'])['transport_cost_usd_dwt_nm']
    .mean()
    .reset_index()
)

# Parse scenario_name → base scenario + ramp-up year
transport_mean[['base_scenario', 'ramp_year']] = transport_mean['scenario_name'].apply(
    lambda x: pd.Series(split_scenario_base_and_ramp_year(x))
)
transport_mean['base_scenario'] = transport_mean['base_scenario'].apply(format_scenario_base_label)

# Assign colours by ramp-up year and linestyles by base scenario
ramp_years_sorted = sorted(transport_mean['ramp_year'].unique())
base_scenarios = sorted(transport_mean['base_scenario'].unique())

cmap = plt.cm.get_cmap('tab10', len(ramp_years_sorted))
year_colors = {yr: cmap(i) for i, yr in enumerate(ramp_years_sorted)}
base_styles = {bs: line_styles[i % len(line_styles)] for i, bs in enumerate(base_scenarios)}

fig, ax = plt.subplots(figsize=(8, 6))

for (bs, yr), grp in transport_mean.groupby(['base_scenario', 'ramp_year']):
    grp_sorted = grp.sort_values('game_year')
    ax.plot(
        grp_sorted['game_year'], grp_sorted['transport_cost_usd_dwt_nm'],
        color=year_colors[yr], linestyle=base_styles[bs], linewidth=2,
        label=f"{bs} – {yr}",
    )

# Build a two-part legend: one for years (color), one for base scenarios (linestyle)
year_handles = [Line2D([], [], color=year_colors[yr], lw=2, label=str(yr)) for yr in ramp_years_sorted]
style_handles = [Line2D([], [], color='grey', lw=2, linestyle=base_styles[bs], label=bs) for bs in base_scenarios]

leg1 = ax.legend(
    handles=style_handles, title='Policy scenario',
    loc='upper center', bbox_to_anchor=(0.5, -0.1),
    ncol=min(2, len(base_scenarios)), frameon=True, fontsize=9, handlelength=4, 
 )
ax.add_artist(leg1)
ax.legend(
    handles=year_handles, title='Biofuel price ramp-up year',
    loc='upper center', bbox_to_anchor=(0.5, -0.3),
    ncol=min(5, len(ramp_years_sorted)), frameon=True, fontsize=8,
)

ax.set_ylabel('Transport cost (USD/dwt-nm)')
ax.set_xlim(2029,2050)
ax.set_title(f"{imo_type}, size bin {size_bin}")
ax.set_xlabel("")
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


# df_fleet_to_plot is not referenced anywhere else after this point; safe to free.    
if CLEAN:
    del df_fleet_to_plot
    gc.collect()

In [ ]:
# Calculate transport cost per ship type, weighted by transport work
# Start fresh from df_fleet (before modifications in cell 76)
df_transport = df_fleet.copy()

# Calculate per-ship transport cost and unit cost
df_transport['transport_cost'] = df_transport[cost_cols_incl_net_policy_cost].sum(axis=1)
df_transport['transport_cost_usd_dwt_nm'] = df_transport['transport_cost'] / df_transport['transport_work_dwt_nm']

# Calculate total transport work for each cohort (this is the weight for averaging)
df_transport['transport_work_cohort'] = df_transport['transport_work_dwt_nm'] * df_transport['real_ships_nb']

# Aggregate by scenario, game_year, and imo_type using weighted average
# Weighted average: sum(value * weight) / sum(weight)
df_transport_agg = (
    df_transport.groupby(['scenario_name', 'game_year', 'imo_type'])
    .apply(lambda g: (g['transport_cost_usd_dwt_nm'] * g['transport_work_cohort']).sum() / g['transport_work_cohort'].sum())
    .reset_index(name='transport_cost_usd_dwt_nm')
    .set_index(['game_year','imo_type'])
)
df_transport_agg_by_size_bin = (
    df_transport.groupby(['scenario_name', 'game_year', 'imo_type','size_id'])
    .apply(lambda g: (g['transport_cost_usd_dwt_nm'] * g['transport_work_cohort']).sum() / g['transport_work_cohort'].sum())
    .reset_index(name='transport_cost_usd_dwt_nm')
    .set_index(['game_year','imo_type'])
)

# Create visualization: one subplot per scenario
n_scenarios = len(scenarios_sub)
n_cols = min(3, n_scenarios)
n_rows = (n_scenarios + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 5*n_rows))

# Flatten axes array for easier iteration
if n_scenarios == 1:
    axes = [axes]
else:
    axes = axes.flatten() if isinstance(axes, np.ndarray) else [axes]

for idx, scenario in enumerate(scenarios_sub):
    ax = axes[idx]
    
    # Pivot for plotting: years as index, ship types as columns
    df_transport_agg[df_transport_agg['scenario_name'] == scenario]['transport_cost_usd_dwt_nm'].unstack().plot(ax=ax, marker='o', linewidth=2)
    
    ax.set_title(format_scenario_base_label(scenario), fontweight='bold')
    ax.set_ylabel('Transport cost ($/dwt-nm)')
    ax.set_xlabel('')
    ax.grid(alpha=0.3)
    ax.legend().remove()
ax.legend(title='Ship type', loc='best', fontsize=8)

# Hide unused subplotns
for idx in range(n_scenarios, len(axes)):
    axes[idx].set_visible(False)

# Add figure-level title
try:
    fig_title = build_ramp_up_years_title(scenarios_sub, prefix='Transport cost by ship type ')
except:
    fig_title = 'Transport cost by ship type'

fig.suptitle(
    fig_title,
    fontsize=12,
    fontweight='bold',
    y=1.00
)

plt.tight_layout()
plt.show()

# Export combined CSV if requested
if SAVE_OUTPUTS:
    # Reset index to get scenario_name column and split it
    df_transport_agg_export = df_transport_agg.reset_index()
    split_parts = df_transport_agg_export['scenario_name'].str.split('_biofuel_ramp_up_', expand=True)
    df_transport_agg_export['policy_scenario'] = split_parts[0]
    df_transport_agg_export['biofuel_scenario'] = split_parts[1]
    
    df_transport_agg_by_size_bin_export = df_transport_agg_by_size_bin.reset_index()
    split_parts_size = df_transport_agg_by_size_bin_export['scenario_name'].str.split('_biofuel_ramp_up_', expand=True)
    df_transport_agg_by_size_bin_export['policy_scenario'] = split_parts_size[0]
    df_transport_agg_by_size_bin_export['biofuel_scenario'] = split_parts_size[1]
    
    export_path = output_csv_dir / f"transport_cost_by_ship_type_v{version_nb}_round_{round_nb}.csv"
    df_transport_agg_export.to_csv(export_path, index=False)
    export_path = output_csv_dir / f"transport_cost_by_ship_type_size_bin_v{version_nb}_round_{round_nb}.csv"
    df_transport_agg_by_size_bin_export.to_csv(export_path, index=False)
    print(f"Exported transport cost data to {export_path}")

# Sensitivity analysis

## Tornado plots

Combined tornado (9 OAT parameters) + boxplot (10 seed replicates) figure for 4 metrics:
cumulative emissions to 2050, year e-fuel uptake passes 10%, ZNZ (dual-fuel) ship share by 2050,
and cumulative cost to 2050. See `ai_training/fem_review_plan.md` section 5 for the design.

In [ ]:
end_year = df_fuels_filtered_annual['estimation_year'].max()
end_year = 2040

# ---- Configuration: fill in scenario_number for each run once available ----
# Metrics are computed from df_fleet / df_rrc_fuel / df_fuel_long, which already hold every
# scenario_* folder found under RUN_OUTPUT_DIR (see the '## Processing' section above) -
# rerun that section with round_nb pointing at the OAT/seed round before running this section.

# Baseline / reference run ("All behavioural biases included"), used both as the
# tornado-chart reference value and as the seed=42 replicate in the boxplot.

# OAT parameters: label -> {'low': scenario_number, 'high': scenario_number}
OAT_PARAMETERS = {
    "Planning horizon":               {"low": "ph-h", "high": "ph-vh"},
    "Peer influence":                 {"low": "pi-l", "high": "pi-h"},
    "Critical mass":                  {"low": "cm-l", "high": "cm-h"},
    "Share of early movers":          {"low": "em-l", "high": "em-h"},
    "Technology maturity threshold":  {"low": "mt-l", "high": "mt-h"},
    "Discount rate":                  {"low": "d-l",  "high": "d-h"},
    "Fuel availability growth rate":  {"low": "fa-l", "high": "fa-h"},
    "e-/gas fuel prices":             {"low": "fp-l", "high": "fp-h"},
    "Carbon price":                   {"low": "cp-l", "high": "cp-h"},
}

# Seed-robustness replicates (baseline seed 42 = baseline above)
SEED_SCENARIOS = {
    101: "rs1", 202: "rs2", 303: "rs3", 404: "rs4", 505: "rs5",
    606: "rs6", 707: "rs7", 808: "rs8", 909: "rs9",
}

METRIC_LABELS = {
    "cumulative_emissions_gt": "Cumulative emissions 2025-2050",
    "efuel_share_end_year": f"Share of e-fuels in {end_year}",
    "df_share_end_year": f"Share of dual-fuel ships in {end_year}",
    "year_efuel_5pct": "Date e-fuel share passes 20% of fuel consumption",
    "year_df_20pct": "Date dual-fuel ship share passes 20% of fleet's dwt",
    "cumulative_cost_bn": "Cumulative cost 2025-2050",
}

METRIC_UNIT = {
    "cumulative_emissions_gt": "GtCO2e",
    "efuel_share_end_year": "% of EJ fuel consumption",
    "df_share_end_year": "% of fleet dwt",
    "year_efuel_5pct": "years",
    "year_df_20pct": "years",
    "cumulative_cost_bn": "billion USD",
}

PERCENT_CHANGE = {
    "cumulative_emissions_gt": True,
    "efuel_share_end_year": False,
    "df_share_end_year": False,
    "year_efuel_5pct": False,
    "year_df_20pct": False,
    "cumulative_cost_bn": True,

}

In [ ]:
# ---- Loader: compute the 4 metrics for a single scenario_number, reusing already-loaded data ----
# (df_fleet, df_rrc_fuel and df_fuel_long are built once in the '## Processing' section above and
# already cover every scenario_* folder under RUN_OUTPUT_DIR - no need to re-open any zip here.)

scenario_number_to_name = (
    df_fleet[['scenario_number', 'scenario_name']]
    .drop_duplicates()
    .set_index('scenario_number')['scenario_name']
)

def load_sensitivity_scenario_metrics(scenario_number):
    """Compute the 4 sensitivity-analysis metrics for one scenario_number, or None if not run yet."""
    scenario_number = str(scenario_number)
    if scenario_number not in scenario_number_to_name.index:
        print(
            f"[sensitivity analysis] scenario_number={scenario_number!r} not found in df_fleet yet - "
            f"skipping (run it and rerun the '## Processing' cells above once available)."
        )
        return None
    scenario_name = scenario_number_to_name.loc[scenario_number]

    df_fleet_s = df_fleet[df_fleet['scenario_number'] == scenario_number]
    df_rrc_s = df_rrc_fuel[df_rrc_fuel['scenario_number'] == scenario_number]
    # df_fuel_long is the clean per-fuel long format (df_split_long + the non-split rows); it only
    # carries scenario_name, not scenario_number, so filter on the name resolved above.
    df_fuel_s = df_fuel_long[df_fuel_long['scenario_name'] == scenario_name]
    assert len(df_fleet_s)> 0
    assert len(df_rrc_s)> 0
    assert len(df_fuel_s)> 0

    # Model horizon end year, taken from the data rather than hardcoded (currently 2049).
    
    # --- Metric 1: cumulative emissions to end_year (MtCO2e) ---
    cumulative_emissions_gt = df_fuels_filtered_annual[
        (df_fuels_filtered_annual.scenario_name == scenario_name)
    ]['tonne_co2e'].sum()

    # --- Metric 2: e-fuel share of fuel consumption in end_year ---
    df_fuel_end_year = df_fuel_s[
        (df_fuel_s['estimation_year'] >= starting_year) & (df_fuel_s['estimation_year'] == end_year)
    ].copy()
    df_fuel_end_year['is_efuel'] = df_fuel_end_year['fuel'].str.contains("e-", case=False, na=False)
    total_consumption_end_year = df_fuel_end_year['fuel_consumption_mj_year_cohort_excl_wind'].sum()
    efuel_share_end_year = (
        df_fuel_end_year.loc[df_fuel_end_year['is_efuel'], 'fuel_consumption_mj_year_cohort_excl_wind'].sum()
        / total_consumption_end_year if total_consumption_end_year > 0 else np.nan
    ) * 100

    # --- Metric 2b: year e-fuel share of fuel consumption passes 20% ---
    df_fuel_all_years = df_fuel_s[df_fuel_s['estimation_year'] <= end_year].copy()
    df_fuel_all_years['is_efuel'] = df_fuel_all_years['fuel'].str.contains("e-", case=False, na=False)
    efuel_share_by_year = df_fuel_all_years.groupby('estimation_year').apply(
        lambda g: g.loc[g['is_efuel'], 'fuel_consumption_mj_year_cohort_excl_wind'].sum()
        / g['fuel_consumption_mj_year_cohort_excl_wind'].sum()
        if g['fuel_consumption_mj_year_cohort_excl_wind'].sum() > 0 else 0
    )
    years_efuel_above_5pct = efuel_share_by_year[efuel_share_by_year >= 0.05]
    year_efuel_5pct = float(years_efuel_above_5pct.index.min()) if len(years_efuel_above_5pct) else np.nan

    # --- Metric 3: year dual-fuel ('DF') ship share passes 10% (share of fleet DWT) ---
    df_fleet_s = df_fleet_s.copy()
    # df_fleet_s['dwt_cohort'] = df_fleet_s['dwt'] * df_fleet_s['real_ships_nb']
    df_fleet_s['is_znz'] = df_fleet_s['engine'].str.contains('DF', case=False, na=False)
    df_znz_by_year = df_fleet_s[df_fleet_s['game_year'] <= end_year].groupby('game_year').apply(
        lambda g: g.loc[g['is_znz'], 'dwt_cohort'].sum() / g['dwt_cohort'].sum()
        if g['dwt_cohort'].sum() > 0 else 0
    )
    years_df_above_10pct = df_znz_by_year[df_znz_by_year >= 0.2]
    year_df_20pct = float(years_df_above_10pct.index.min()) if len(years_df_above_10pct) else np.nan

    # --- Metric 3b: dual-fuel ('DF') ship share of fleet DWT in end_year ---
    df_fleet_end_year = df_fleet_s[df_fleet_s['game_year'] == end_year]
    total_dwt_end_year = df_fleet_end_year['dwt_cohort'].sum()
    df_share_end_year = (
        df_fleet_end_year.loc[df_fleet_end_year['is_znz'], 'dwt_cohort'].sum() / total_dwt_end_year
        if total_dwt_end_year > 0 else np.nan
    ) * 100

    # --- Metric 4: cumulative cost to end_year (billion USD) ---
    # Fleet-level capex/opex/stranded value/fuel cost, excluding the policy fund and social cost of
    # carbon terms (which require the multi-scenario SU-market fund pipeline built in the "## Fund"
    # section above and are out of scope for this single-scenario OAT/seed metric).
    cumulative_cost_bn = cum_cost_usd.loc[scenario_name,:].sum() * pow(10, -9)

    return {
        "cumulative_emissions_gt": cumulative_emissions_gt,
        "efuel_share_end_year": efuel_share_end_year,
        "df_share_end_year": df_share_end_year,
        "year_efuel_5pct": year_efuel_5pct,
        "year_df_20pct": year_df_20pct,
        "cumulative_cost_bn": cumulative_cost_bn,
    }

In [ ]:
# ---- Compute metrics for baseline, each OAT low/high, and each seed replicate ----
assert BASELINE_SCENARIO_NUMBER is not None, (
    "Set BASELINE_SCENARIO_NUMBER before running the sensitivity analysis."
)
assert bau_label is not None, (
    "Set bau_label before running the sensitivity analysis."
)

baseline_metrics = load_sensitivity_scenario_metrics(BASELINE_SCENARIO_NUMBER)

oat_results = {}
for param_label, levels in OAT_PARAMETERS.items():
    oat_results[param_label] = {
        level: load_sensitivity_scenario_metrics(scenario_number)
        for level, scenario_number in levels.items()
    }

seed_results = {42: baseline_metrics}
for seed, scenario_number in SEED_SCENARIOS.items():
    seed_results[seed] = load_sensitivity_scenario_metrics(scenario_number)

# Drop OAT parameters/seeds whose scenario hasn't been run yet, so plotting can proceed with what's available.
oat_results = {
    label: levels for label, levels in oat_results.items()
    if levels['low'] is not None and levels['high'] is not None
}
seed_results = {seed: metrics for seed, metrics in seed_results.items() if metrics is not None}

print("Baseline metrics:", baseline_metrics)
print(f"OAT parameters available: {list(oat_results.keys())}")
print(f"Seed replicates available: {sorted(seed_results.keys())}")

In [ ]:
# ---- Combined tornado + seed boxplot figure, one axes per metric, both expressed as ----
# ---- deviation from the baseline value so the two are directly comparable. ----
metric_keys = [
    "efuel_share_end_year",
    "df_share_end_year",
    # "year_efuel_5pct",
    # "year_df_20pct",
    "cumulative_emissions_gt",
    "cumulative_cost_bn",
]
n_metrics = len(metric_keys)

fig, axes = plt.subplots(n_metrics, 1, figsize=(8, 3.2 * n_metrics), squeeze=False)
axes = axes[:, 0]

assert baseline_metrics is not None, "Baseline scenario hasn't been run yet - see the print message above."
param_labels = list(oat_results.keys())
seeds_sorted = sorted(seed_results.keys())

for row, metric_key in enumerate(metric_keys):
    ax = axes[row]
    baseline_val = baseline_metrics[metric_key]

    # --- Tornado: impact of each OAT parameter's low/high vs baseline ---
    deltas = []
    for param_label in param_labels:
        low_val = min(oat_results[param_label]['low'][metric_key], baseline_val)
        high_val = max(oat_results[param_label]['high'][metric_key], baseline_val)
        if PERCENT_CHANGE[metric_key]:
            low_val = (low_val - baseline_val) * 100 / baseline_val
            high_val = (high_val - baseline_val) * 100 / baseline_val
        else:
            low_val = low_val - baseline_val
            high_val = high_val - baseline_val
        impact = abs(high_val - low_val) if pd.notna(high_val) and pd.notna(low_val) else 0
        deltas.append((param_label, low_val, high_val, impact))

    deltas.sort(key=lambda d: d[3])
    n_params = len(deltas)
    y_pos = np.arange(n_params)

    for i, (param_label, low_val, high_val, impact) in enumerate(deltas):
        left = min(low_val, high_val)
        width = abs(high_val - low_val)
        ax.barh(i, width, left=left, color='steelblue', edgecolor='black')

    # --- Seed boxplot: same deviation-from-baseline scale, placed above the tornado bars ---
    seed_box_y = n_params + 1
    seed_deltas = [seed_results[s][metric_key] - baseline_val for s in seeds_sorted]
    if PERCENT_CHANGE[metric_key]:
        seed_deltas = seed_deltas * 100 / baseline_val
    ax.boxplot(seed_deltas, positions=[seed_box_y], vert=False, widths=0.6)

    ax.axvline(0, color='black', linewidth=1)
    ax.axhline(n_params + 0.5, color='grey', linewidth=0.8, linestyle=':')
    ax.set_yticks(list(y_pos) + [seed_box_y])
    ax.set_yticklabels([d[0] for d in deltas] + ['Seed replicates (10 runs)'], fontsize=8)
    ax.set_title(METRIC_LABELS[metric_key], fontsize=12)

    ax.set_xlabel(f"Deviation from baseline ({"% change in " if PERCENT_CHANGE[metric_key] else ""}{METRIC_UNIT[metric_key]})", fontsize=8)
    ax.grid(alpha=0.3, axis='x')
    if row == 0:
        ax.legend(loc='best', fontsize=8)

plt.tight_layout()
plt.show()

if SAVE_OUTPUTS:
    fig.savefig(figures_dir / f"sensitivity_analysis_v{version_nb}_round_{round_nb}.png", dpi=300, bbox_inches='tight')